In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/CIS 5190: Final Project') # Replace 'MyDrive' with the actual path if different
# os.chdir('/content/drive/MyDrive/Fall2024/CIS5190AppliedMachineLearning/CIS 5190: Final Project') # Replace 'MyDrive' with the actual path if different
# Now your current working directory is set to the directory containing this .ipynb file

Mounted at /content/drive


# General Suggestions

- Maintain detailed notes on the preprocessing steps applied.

In [ ]:
#!pip install nltk - run this and restart runtime

In [ ]:
# Pre-processing and cleaning
import pandas as pd
import numpy as np

In [ ]:
# Imports:

import string
from nltk.corpus import stopwords
import nltk
from nltk.stem import WordNetLemmatizer

# Machine learning
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score

# Additional Models
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, GlobalMaxPooling1D, Dense, Input, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

# 3.1 Problem Definition and Motivation
- Scrape the News Headlines dataset and train on a Binary Text Classification task using news headlines from two prominent news outlets:
  - Fox News and NBC News.
- Goal: Create some machine-learning models that classify news based on their headlines.
  - We are given a baseline model with lower accuracies, 3815 URLs of 3815 news (so 2010 from FoxNews, 1805 from NBC) for you to start Headlines scraping, and a screenshot of the first few lines of hidden test data that we will use to test your models. Our task is to collect a dataset, process it, and experiment with various models to improve classification performance as best as you can. To show and summarize your improvement, in the final report, you will submit one or several line charts of your models’ metrics, including metrics of the baseline model.
  - Dataset of links to articles from NBC News and Fox News: https://drive.google.com/file/d/1-5JbZECwn5VLgo-Uk7_94w2eEI2R4A-u/view?usp=sharing


## 3.1.1 Performance metrics
For your final model performance, we will evaluate your models on news headlines scrapped from NBC News and Fox News after the deadline submission. As such, your model will not have access to the testing set when you are training/validating your model. In the sections below, we describe what the data looks like and give you tips on how to clean the data, as well as how to do the webscrapping.

# 3.3 Code for Training a Baseline Model

- I think we can delete this cell below since we did a baseline model after this section with the same process

In [ ]:
# 1. Load the CSV files & Preprocess the data
# csv_file_path = ’/merged_news_data.csv’
# news _df = pd.read_csv(csv_filePath)
# ... ...
# 2. Split the data into training and testing sets
# (80% train, 20% test)
# ... ...
# 3. Convert the labels to binary values (0 for ’FoxNews’, 1 for ’NBC’)
y_train = y_train.apply(lambda x: 1 if x == ’FoxNews’ else 0)
y_test = y_test.apply(lambda x: 1 if x == ’FoxNews’ else 0)
# 4. Convert the text data to TF-IDF features
vectorizer = TfidfVectorizer(stop_words=’english’, max_features=100)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)
# 5. Train a Logistic Regression model
model = LogisticRegression(max_iter=100)
model.fit(X_train_tfidf, y_train)
# 6. Make predictiosn on the test set
y_pred = model.predict(X_test_tfidf)
# 7. Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")
print("Classification Report:\n", classification_report(y_test, y_pred))

## Result
# Accuracy: 0.6649
# Classification Report:
# # #
#    precision
# 0    0.69
# 1 0.65
# recall   f1-score
#  0.54      0.60       358
#  0.78      0.71       400
# support
#
#     accuracy
#    macro avg
# weighted avg
# 0.67       0.66
# 0.67       0.66
# 0.66       758
# 0.66       758
# 0.66       758

SyntaxError: invalid character '’' (U+2019) (<ipython-input-3-805d74839f16>, line 9)

# 3.4 Python Web Scraping
The Python libraries that are most helpful are BeautifulSoup and request. To web scrape, you identify the HTML tabs that contain the text that you want. In the example below, we are scrapping the headline which is in a \<h1\> tag with the class name "headline speakable." Here is a sample Python code to web scrape:

In [ ]:
import requests
from bs4 import BeautifulSoup
url = 'https://www.foxnews.com/sports/juan-soto-sends-yankees-world-series-first-time-15-years' # example website
response = requests.get(url)
if response.status_code != 200:
  raise Exception(f"Failed to load page: Status code {response.status_code}")
# Parse the HTML content with BeautifulSoup
soup = BeautifulSoup(response.text, "html.parser")
title = soup.find("h1", class_="headline speakable").get_text()
# title should be the following
# Juan Soto sends the Yankees to the World Series for the first time in 15 years

In [ ]:
print(title)

Juan Soto sends the Yankees to the World Series for the first time in 15 years


In [ ]:
import pandas as pd
url_file = 'url_only.csv'
urls_df = pd.read_csv(url_file)

# Add empty columns for titles and sources
urls_df['title'] = ''
urls_df['source'] = ''

# Loop through each URL in the DataFrame
for index, row in urls_df.iterrows():
    url = row['url']
    try:
        # Send an HTTP request to the URL
        response = requests.get(url)
        if response.status_code != 200:
            raise Exception(f"Failed to load page: Status code {response.status_code}")

        # Parse the HTML content
        soup = BeautifulSoup(response.text, "html.parser")

        # Determine the news source from the URL
        if "foxnews.com" in url:
            source = "Fox News"
            title_tag = soup.find("h1", class_="headline speakable") or soup.find("h1")
        elif "nbcnews.com" in url:
            source = "NBC News"
            title_tag = soup.find("h1")
        else:
            source = "Other"
            title_tag = soup.find("title") or soup.find("h1")

        # Extract the title
        title = title_tag.get_text(strip=True) if title_tag else "Title not found"

        # Store title and source in the DataFrame
        urls_df.at[index, 'title'] = title
        urls_df.at[index, 'source'] = source

    except Exception as e:
        # If there's an error, store error message in title and source columns
        urls_df.at[index, 'title'] = "Error retrieving title"
        urls_df.at[index, 'source'] = str(e)

# Save the updated DataFrame to a new CSV file
urls_df.to_csv("titles_and_sources.csv", index=False)
print("Results saved to titles_and_sources.csv")

Results saved to titles_and_sources.csv


Fox News Title Extraction

In [ ]:
import pandas as pd
fox_url_file = 'combined_no_duplicates.csv'
urls_df = pd.read_csv(fox_url_file)

# Add empty columns for titles and sources
urls_df['title'] = ''
urls_df['source'] = ''

# Loop through each URL in the DataFrame
for index, row in urls_df.iterrows():
    url = row['Article URL']
    try:
        # Send an HTTP request to the URL
        response = requests.get(url)
        if response.status_code != 200:
            raise Exception(f"Failed to load page: Status code {response.status_code}")

        # Parse the HTML content
        soup = BeautifulSoup(response.text, "html.parser")

        # Determine the news source from the URL
        source = "Fox News"
        title_tag = soup.find("h1", class_="headline speakable") or soup.find("h1")

        # Extract the title
        title = title_tag.get_text(strip=True) if title_tag else "Title not found"

        # Store title and source in the DataFrame
        urls_df.at[index, 'title'] = title
        urls_df.at[index, 'source'] = source

    except Exception as e:
        # If there's an error, store error message in title and source columns
        urls_df.at[index, 'title'] = "Error retrieving title"
        urls_df.at[index, 'source'] = str(e)

# Save the updated DataFrame to a new CSV file
urls_df.to_csv("fox_titles.csv", index=False)
print("Results saved to fox_titles.csv")

Results saved to fox_titles.csv


In [ ]:
titles_and_sources.csv

NameError: name 'titles_and_sources' is not defined

In [ ]:
urls_df

,url,title,source
0,https://www.foxnews.com/lifestyle/jack-carrs-e...,Jack Carr recalls Gen. Eisenhower's D-Day memo...,Fox News
1,https://www.foxnews.com/entertainment/bruce-wi...,"Bruce Willis, Demi Moore avoided doing one thi...",Fox News
2,https://www.foxnews.com/politics/blinken-meets...,Error retrieving title,'NoneType' object has no attribute 'get_text'
3,https://www.foxnews.com/entertainment/emily-bl...,Emily Blunt says her ‘toes curl’ when people t...,Fox News
4,https://www.foxnews.com/media/the-view-co-host...,"'The View' co-host, CNN commentator Ana Navarr...",Fox News
...,...,...,...
3800,https://www.nbcnews.com/politics/2024-election...,Error retrieving title,'NoneType' object has no attribute 'get_text'
3801,https://www.nbcnews.com/select/shopping/best-a...,Error retrieving title,'NoneType' object has no attribute 'get_text'
3802,https://www.nbcnews.com/select/shopping/best-v...,Error retrieving title,'NoneType' object has no attribute 'get_text'
3803,https://www.nbcnews.com/politics/2024-election...,Error retrieving title,'NoneType' object has no attribute 'get_text'


# 3.4.1 Cleaning Process Suggestions
After scraping the headlines, consider further cleaning to make sure your data is ready for training. Some possible operations are listed, but they are not required or no guarantee of performance improvement. You are highly encouraged to explore on your own and incorporate more advanced techniques:


11/30:

Importing data (provided by teaching team + our own data):


In [ ]:
# prompt: what is my pwd? I have the folder titles_and_sources.csv in my drive but dont know how to get that to be the pwd and how to load that file. the files are just in my drive, not in any other subfolder

import pandas as pd
# Assuming titles_and_sources.csv is in your Google Drive's root directory
file_path = '/content/drive/MyDrive/titles_and_sources.csv'  # Update with correct path if needed

try:
    df = pd.read_csv(file_path)
    print(df.head())  # Print first few rows to verify
except FileNotFoundError:
    print(f"Error: File not found at {file_path}")
except pd.errors.EmptyDataError:
    print(f"Error: File at {file_path} is empty")
except pd.errors.ParserError:
    print(f"Error: Could not parse file at {file_path}. Check file format")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Error: File not found at /content/drive/MyDrive/titles_and_sources.csv


# START HERE: Post Data Collection:

Run the code below once the data has been saved.

In [ ]:
news_title_and_source = pd.read_csv("titles_and_sources.csv")
extra_nbc_titles = pd.read_csv("all_nbc_titles.csv")
extra_fox_titles = pd.read_csv("all_fox_sources.csv")

# Part 1: Data Processing

## 1.1 Exploring Datasets

We have 3 datasets. The first is titled `news_title_and_source`, the second is `extra_nbc_titles` which consists of the additional NBC article titles that we scraped, and the third is `extra_fox_titles`. Below, we will explore the three datasets, observing the number of rows for each.

### Dataset 1: `news_title_and_source`

In [ ]:
news_title_and_source.head()

,url,title,source
0,https://www.foxnews.com/lifestyle/jack-carrs-e...,Jack Carr recalls Gen. Eisenhower's D-Day memo...,Fox News
1,https://www.foxnews.com/entertainment/bruce-wi...,"Bruce Willis, Demi Moore avoided doing one thi...",Fox News
2,https://www.foxnews.com/politics/blinken-meets...,Error retrieving title,'NoneType' object has no attribute 'get_text'
3,https://www.foxnews.com/entertainment/emily-bl...,Emily Blunt says her ‘toes curl’ when people t...,Fox News
4,https://www.foxnews.com/media/the-view-co-host...,"'The View' co-host, CNN commentator Ana Navarr...",Fox News


In [ ]:
news_title_and_source.shape

(3805, 3)

In [ ]:
news_title_and_source['source'].value_counts()

,count
source,
'NoneType' object has no attribute 'get_text',2256
Fox News,1547
NBC News,1
Failed to load page: Status code 500,1


In [ ]:
news_title_and_source.isna().sum()

,0
url,0
title,0
source,0


### Dataset 2: `extra_nbc_titles`

In [ ]:
extra_nbc_titles.head()

,Category,Article URL,title,source
0,health,https://www.nbcnews.com/nightly-news/video/stu...,Study suggests more lives could be saved with ...,NBC News
1,business,https://www.nbcnews.com/nbc-out/out-pop-cultur...,Singer Khalid comes out as gay after he was outed,NBC News
2,sports,https://www.nbcnews.com/politics/congress/-not...,'He is not Mitt Romney and he is not Donald Tr...,NBC News
3,tech-media,https://www.nbcnews.com/news/world/climate-act...,Rich nations raise COP29 climate finance offer...,NBC News
4,world,https://www.nbcnews.com/news/world/emperor-pen...,"'Good luck, Gus': Emperor penguin found in Aus...",NBC News


In [ ]:
extra_nbc_titles.shape

(1083, 4)

In [ ]:
extra_nbc_titles = extra_nbc_titles.drop_duplicates()
extra_nbc_titles.shape

(1083, 4)

In [ ]:
extra_nbc_titles.isna().sum()

,0
Category,0
Article URL,0
title,1
source,0


### Dataset 3: `extra_fox_titles`

In [ ]:
extra_fox_titles.head()

,Category,Article URL,Subcategory,title,source
0,us,https://www.foxnews.com/us/southwest-airlines-...,NaN,Southwest Airlines flight struck by bullet pri...,Fox News
1,us,https://www.foxnews.com/us/california-inmate-a...,NaN,California inmate accused of killing cellmate ...,Fox News
2,us,https://www.foxnews.com/us/north-carolina-gun-...,NaN,North Carolina gun laws would strengthen under...,Fox News
3,us,https://www.foxnews.com/us/friend-who-heard-mu...,NaN,Friend who heard murder confession thought fur...,Fox News
4,us,https://www.foxnews.com/us/florida-burglary-vi...,NaN,Florida burglary victim arrested after police ...,Fox News


In [ ]:
extra_fox_titles.shape

(1994, 5)

In [ ]:
extra_fox_titles.isna().sum()

,0
Category,0
Article URL,0
Subcategory,772
title,1307
source,1307


## 1.2 Merging Datasets:
We will now combine all of our three datasets into one dataset, called `combined_data`

Combining just the (Article) URLs, the title and the sources:

In [ ]:
# Extract relevant columns
news_title_and_source_subset = news_title_and_source[['url', 'title', 'source']]
extra_nbc_titles_subset = extra_nbc_titles[['Article URL', 'title', 'source']].rename(columns={'Article URL': 'url'})
extra_fox_titles_subset = extra_fox_titles[['Article URL', 'title', 'source']].rename(columns={'Article URL': 'url'})

# Concatenate the datasets
combined_data = pd.concat([news_title_and_source_subset, extra_nbc_titles_subset, extra_fox_titles_subset], ignore_index=True)

combined_data.head()

,url,title,source
0,https://www.foxnews.com/lifestyle/jack-carrs-e...,Jack Carr recalls Gen. Eisenhower's D-Day memo...,Fox News
1,https://www.foxnews.com/entertainment/bruce-wi...,"Bruce Willis, Demi Moore avoided doing one thi...",Fox News
2,https://www.foxnews.com/politics/blinken-meets...,Error retrieving title,'NoneType' object has no attribute 'get_text'
3,https://www.foxnews.com/entertainment/emily-bl...,Emily Blunt says her ‘toes curl’ when people t...,Fox News
4,https://www.foxnews.com/media/the-view-co-host...,"'The View' co-host, CNN commentator Ana Navarr...",Fox News


Let us see how many titles we have from each source:

In [ ]:
combined_data['source'].value_counts()

,count
source,
'NoneType' object has no attribute 'get_text',2256
Fox News,2234
NBC News,1084
Failed to load page: Status code 500,1


In [ ]:
filtered_data = combined_data.dropna(subset=['source', 'title'])
source_counts = filtered_data['source'].value_counts()

source_counts

,count
source,
'NoneType' object has no attribute 'get_text',2256
Fox News,2234
NBC News,1083
Failed to load page: Status code 500,1


# Part 2: Data Cleaning

We will now clean our `combined_data` dataframe. We will do the following:
- Remove unwanted elements such as HTML tags, scripts, and special characters.
- Address unnecessary spaces, tabs, and newline characters to ensure consistency.
- Decide how to handle punctuation marks within the headlines.


## 2.1 Handling Missing or Incomplete Data:
Detect any missing or incomplete headlines in your dataset. Choose methods to handle these gaps, such as removing incomplete entries or imputing missing values.

In [ ]:
# We already extracted the news titled above
# Let us drop the nulls since we cannot use the data item for training if it has either the source or the title missing
combined_data_usable = combined_data.dropna()
combined_data_usable.isna().sum()

,0
url,0
title,0
source,0


In [ ]:
combined_data_usable

,url,title,source
0,https://www.foxnews.com/lifestyle/jack-carrs-e...,Jack Carr recalls Gen. Eisenhower's D-Day memo...,Fox News
1,https://www.foxnews.com/entertainment/bruce-wi...,"Bruce Willis, Demi Moore avoided doing one thi...",Fox News
2,https://www.foxnews.com/politics/blinken-meets...,Error retrieving title,'NoneType' object has no attribute 'get_text'
3,https://www.foxnews.com/entertainment/emily-bl...,Emily Blunt says her ‘toes curl’ when people t...,Fox News
4,https://www.foxnews.com/media/the-view-co-host...,"'The View' co-host, CNN commentator Ana Navarr...",Fox News
...,...,...,...
5570,https://www.foxnews.com/lifestyle/fall-activit...,Bundle up for your favorite fall activities wi...,Fox News
5571,https://www.foxnews.com/lifestyle/home-decor-r...,Reusable décor to design your home for the win...,Fox News
5572,https://www.foxnews.com/us/murdaugh-hunting-es...,Murdaugh hunting estate buyer says it will loo...,Fox News
5573,https://www.foxnews.com/lifestyle/halloween-po...,Halloween porch pumpkins business brings in ov...,Fox News


## 2.2 Consistency Checks:
Ensure all headlines follow a consistent format. Identify and address duplicate headlines to maintain the integrity of your dataset.

In [ ]:
combined_data_usable = combined_data_usable.drop_duplicates()
combined_data_usable.shape

(4833, 3)

In [ ]:
combined_data_usable['source'].value_counts()

,count
source,
'NoneType' object has no attribute 'get_text',2256
Fox News,2145
NBC News,431
Failed to load page: Status code 500,1


In [ ]:
combined_data_usable = combined_data_usable[combined_data_usable['source'].isin(["Fox News", "NBC News"])]

In [ ]:
combined_data_usable['source'].value_counts()

,count
source,
Fox News,2145
NBC News,431


**Now, after dropping nulls and duplicates, we have 2598 Fox news titles and 2233 NBC titles.**

## 2.3 Normalization:
- Standardize the text by converting it to lowercase or uppercase.
- Remove punctuation.
- Remove leading and trailing spaces.
- Remove common stopwords that may not add significant value.
- Apply stemming or lemmatization to reduce words to their base forms.

In [ ]:
# Standardizing the title by convering it to lowercase
combined_data_usable['title_lower'] = combined_data_usable['title'].str.lower()

# Removing punctuation
# string.punctuation is a predefined string in python containing all common punctuation marks: !"#$%&'()*+,-./:;<=>?@[\]^_{|}~
# str.replace replaces all characters that match the punctation marks from string.punctuation (with nothing)
import string
combined_data_usable['title_lower'] = combined_data_usable['title_lower'].str.replace(f"[{string.punctuation}]", "", regex=True)

# Removing leading and trailing spaces
combined_data_usable['title_lower'] = combined_data_usable['title_lower'].str.strip()

combined_data_usable

<ipython-input-23-316e3638b6b3>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  combined_data_usable['title_lower'] = combined_data_usable['title'].str.lower()
<ipython-input-23-316e3638b6b3>:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  combined_data_usable['title_lower'] = combined_data_usable['title_lower'].str.replace(f"[{string.punctuation}]", "", regex=True)
<ipython-input-23-316e3638b6b3>:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .lo

,url,title,source,title_lower
0,https://www.foxnews.com/lifestyle/jack-carrs-e...,Jack Carr recalls Gen. Eisenhower's D-Day memo...,Fox News,jack carr recalls gen eisenhowers dday memo ab...
1,https://www.foxnews.com/entertainment/bruce-wi...,"Bruce Willis, Demi Moore avoided doing one thi...",Fox News,bruce willis demi moore avoided doing one thin...
2,https://www.foxnews.com/politics/blinken-meets...,"Blinken meets Qatar PM, says Israeli actions a...",Fox News,blinken meets qatar pm says israeli actions ar...
3,https://www.foxnews.com/entertainment/emily-bl...,Emily Blunt says her ‘toes curl’ when people t...,Fox News,emily blunt says her ‘toes curl’ when people t...
4,https://www.foxnews.com/media/the-view-co-host...,"'The View' co-host, CNN commentator Ana Navarr...",Fox News,the view cohost cnn commentator ana navarro to...
...,...,...,...,...
5569,https://www.foxnews.com/video/6363517795112,Check out these easy and affordable home updat...,Fox News,check out these easy and affordable home updat...
5570,https://www.foxnews.com/lifestyle/fall-activit...,Bundle up for your favorite fall activities wi...,Fox News,bundle up for your favorite fall activities wi...
5571,https://www.foxnews.com/lifestyle/home-decor-r...,Reusable décor to design your home for the win...,Fox News,reusable décor to design your home for the win...
5572,https://www.foxnews.com/us/murdaugh-hunting-es...,Murdaugh hunting estate buyer says it will loo...,Fox News,murdaugh hunting estate buyer says it will loo...


The column `title_lower` has all titles in lowercase, with their punctuation removed and stripped of leading and trailing spaces.

## 2.4 Removing stopwords:

**Stopwords** are common words in a language that are typically filtered out in text analysis because they don't carry significant meaning on their own. They are words that occur frequently in language but are often not useful for understanding the main content of a text or for natural language processing (NLP) tasks. These words include articles, conjunctions, prepositions, and auxiliary verbs, which don't add specific information in context.

### Examples of Stopwords (in English):
- **Articles**: the, a, an
- **Conjunctions**: and, or, but, because
- **Prepositions**: in, on, at, by, with
- **Auxiliary Verbs**: is, are, was, be, have
- **Pronouns**: he, she, they, it, we, you
- **Other Common Words**: of, for, to, from, about, as, that

#### Why Remove Stopwords?
- **Efficiency**: Removing stopwords helps reduce the size of the dataset, making processing faster.
- **Noise Reduction**: These words don’t typically contribute to understanding the meaning of a sentence, especially in tasks like text classification, sentiment analysis, or topic modeling.
- **Improved Model Accuracy**: Models may perform better by focusing on meaningful words (e.g., nouns and verbs) that actually help classify or understand the content.

#### Example:
Consider the sentence:  
**"The cat sat on the mat."**

- **Stopwords**: "the" (appears twice), "on"  
- **Useful Words**: "cat", "sat", "mat"

After removing stopwords, the meaningful content remains:  
**"cat sat mat"**

### How Stopwords Are Used in NLP
Stopwords are typically removed when processing text data for various tasks, such as:
- **Text Classification**: Understanding whether an email is spam or not.
- **Sentiment Analysis**: Determining if a product review is positive or negative.
- **Topic Modeling**: Identifying the main topics in a collection of documents.

By eliminating stopwords, the models focus on the **relevant keywords** that help in making predictions or gaining insights.

In [ ]:
#!pip install nltk - run this and restart runtime

In [ ]:
# Downloading stop words
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')

# Define stopwords
stop_words = set(stopwords.words('english'))

# Function to remove stopwords from preprocessed title_lower
def remove_stopwords(text):
    if isinstance(text, str):  # The input is a string
        words = text.split()  # Split into words
        filtered_words = [word for word in words if word not in stop_words]
        return " ".join(filtered_words)
    return text  # Return as-is if not a string

# Apply the function to the 'title_lower' column
combined_data_usable['title_cleaned'] = combined_data_usable['title_lower'].apply(remove_stopwords)

combined_data_usable.head()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
<ipython-input-25-e4d1252550b7>:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  combined_data_usable['title_cleaned'] = combined_data_usable['title_lower'].apply(remove_stopwords)


,url,title,source,title_lower,title_cleaned
0,https://www.foxnews.com/lifestyle/jack-carrs-e...,Jack Carr recalls Gen. Eisenhower's D-Day memo...,Fox News,jack carr recalls gen eisenhowers dday memo ab...,jack carr recalls gen eisenhowers dday memo gr...
1,https://www.foxnews.com/entertainment/bruce-wi...,"Bruce Willis, Demi Moore avoided doing one thi...",Fox News,bruce willis demi moore avoided doing one thin...,bruce willis demi moore avoided one thing copa...
2,https://www.foxnews.com/politics/blinken-meets...,"Blinken meets Qatar PM, says Israeli actions a...",Fox News,blinken meets qatar pm says israeli actions ar...,blinken meets qatar pm says israeli actions re...
3,https://www.foxnews.com/entertainment/emily-bl...,Emily Blunt says her ‘toes curl’ when people t...,Fox News,emily blunt says her ‘toes curl’ when people t...,emily blunt says ‘toes curl’ people tell kids ...
4,https://www.foxnews.com/media/the-view-co-host...,"'The View' co-host, CNN commentator Ana Navarr...",Fox News,the view cohost cnn commentator ana navarro to...,view cohost cnn commentator ana navarro host n...


The column `title_cleaned` has all titles in lowercase, with their punctuation removed, stripped of leading and trailing spaces, and stripped of stop words.

## 2.5 Lemmatization

Lemmatization is the process of reducing a word to its base or dictionary form, known as its "lemma," by considering the word's meaning and its grammatical context. Unlike stemming, which simply chops off prefixes or suffixes, lemmatization ensures that the resulting word is a valid word in the language. For example, "running" becomes "run," and "better" becomes "good." This is done using linguistic rules and dictionaries, making lemmatization more accurate in preserving the word's meaning. It's particularly useful in natural language processing tasks like text classification or information retrieval, where preserving the correct form of a word helps improve model performance.








In [ ]:
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')

# Create lemmatizer object
lemmatizer = WordNetLemmatizer()

# Function to lemmatize each word
def lemmatize_title(text):
    if isinstance(text, str):  # Ensure it's a string
        words = text.split()
        lemmatized_words = [lemmatizer.lemmatize(word, pos='v') for word in words]  # Assuming verbs for action words
        return " ".join(lemmatized_words)
    return text

# Apply lemmatization to 'title_lower' column
combined_data_usable['title_lemmatized'] = combined_data_usable['title_cleaned'].apply(lemmatize_title)

combined_data_usable.head()

[nltk_data] Downloading package wordnet to /root/nltk_data...
<ipython-input-26-87562814208e>:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  combined_data_usable['title_lemmatized'] = combined_data_usable['title_cleaned'].apply(lemmatize_title)


,url,title,source,title_lower,title_cleaned,title_lemmatized
0,https://www.foxnews.com/lifestyle/jack-carrs-e...,Jack Carr recalls Gen. Eisenhower's D-Day memo...,Fox News,jack carr recalls gen eisenhowers dday memo ab...,jack carr recalls gen eisenhowers dday memo gr...,jack carr recall gen eisenhowers dday memo gre...
1,https://www.foxnews.com/entertainment/bruce-wi...,"Bruce Willis, Demi Moore avoided doing one thi...",Fox News,bruce willis demi moore avoided doing one thin...,bruce willis demi moore avoided one thing copa...,bruce willis demi moore avoid one thing copare...
2,https://www.foxnews.com/politics/blinken-meets...,"Blinken meets Qatar PM, says Israeli actions a...",Fox News,blinken meets qatar pm says israeli actions ar...,blinken meets qatar pm says israeli actions re...,blinken meet qatar pm say israeli action retal...
3,https://www.foxnews.com/entertainment/emily-bl...,Emily Blunt says her ‘toes curl’ when people t...,Fox News,emily blunt says her ‘toes curl’ when people t...,emily blunt says ‘toes curl’ people tell kids ...,emily blunt say ‘toes curl’ people tell kid wa...
4,https://www.foxnews.com/media/the-view-co-host...,"'The View' co-host, CNN commentator Ana Navarr...",Fox News,the view cohost cnn commentator ana navarro to...,view cohost cnn commentator ana navarro host n...,view cohost cnn commentator ana navarro host n...


The column `title_lemmatized` has all titles in lowercase, with their punctuation removed, stripped of leading and trailing spaces, stripped of stop words and titles lemmatized.

## 2.5 Data Validation:
We will now manually inspect a subset of processed headlines for quality assurance. Calculate metrics such as average headline length or word frequency distributions to understand your dataset better.

In [ ]:
# Calculate the length of each headline in characters
combined_data_usable.loc[:,'headline_length'] = combined_data_usable['title_lemmatized'].apply(len)

# Calculate the length of each headline in terms of words
combined_data_usable.loc[:,'word_count'] = combined_data_usable['title_lemmatized'].apply(lambda x: len(x.split()))

# Calculate the average headline length in characters and words
avg_headline_length = combined_data_usable['headline_length'].mean()
avg_word_count = combined_data_usable['word_count'].mean()

print(f"Overall: Average Headline Length (Characters): {avg_headline_length}")
print(f"Overall: Average Word Count: {avg_word_count}")

Overall: Average Headline Length (Characters): 64.64452720877301
Overall: Average Word Count: 9.618663356093524


<ipython-input-27-2cf99cdde82a>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  combined_data_usable.loc[:,'headline_length'] = combined_data_usable['title_lemmatized'].apply(len)
<ipython-input-27-2cf99cdde82a>:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  combined_data_usable.loc[:,'word_count'] = combined_data_usable['title_lemmatized'].apply(lambda x: len(x.split()))


In [ ]:
# Remove rows with invalid sources (e.g., URLs or error messages)
valid_sources = ['Fox News', 'NBC News']  # Add any other valid sources as needed

# Filter the DataFrame to only include valid sources
combined_data_usable_cleaned = combined_data_usable[combined_data_usable['source'].isin(valid_sources)]

# Now calculate the average headline length and word count for each source
avg_headline_length_by_source = combined_data_usable_cleaned.groupby('source')['headline_length'].mean()
avg_word_count_by_source = combined_data_usable_cleaned.groupby('source')['word_count'].mean()

# Avg lengths in terms of character
print("Average Headline Length by Source:")
print(avg_headline_length_by_source)

# Avg word count
print("\nAverage Word Count by Source:")
print(avg_word_count_by_source)

Average Headline Length by Source:
source
Fox News    69.451886
NBC News    59.091357
Name: headline_length, dtype: float64

Average Word Count by Source:
source
Fox News    10.289453
NBC News     8.844156
Name: word_count, dtype: float64


In [ ]:
combined_data_usable_cleaned[['title_lemmatized', 'source']]

,title_lemmatized,source
0,jack carr recall gen eisenhowers dday memo gre...,Fox News
1,bruce willis demi moore avoid one thing copare...,Fox News
2,blinken meet qatar pm say israeli action retal...,Fox News
3,emily blunt say ‘toes curl’ people tell kid wa...,Fox News
4,view cohost cnn commentator ana navarro host n...,Fox News
...,...,...
5569,check easy affordable home update holiday,Fox News
5570,bundle favorite fall activities outdoor apparel,Fox News
5571,reusable décor design home winter also work su...,Fox News
5572,murdaugh hunt estate buyer say look completely...,Fox News


In [ ]:
combined_data_usable_cleaned['source'].unique()

array(['Fox News', 'NBC News'], dtype=object)

# Part 3: Machine Learning

# Part 3: Given Models

Below, we have 2 models: a majority class predictor and a baseline logistic regression model.

In [ ]:
# import train and test sets (pre-cleaned):
train_df = pd.read_csv('train_data.csv')
test_df = pd.read_csv('test_data.csv')

# change target variables (source) into categorical with feature mapping:
source_mapping = {'Fox News': 0, 'NBC News': 1}
train_df['source'] = train_df['source'].map(source_mapping)
test_df['source'] = test_df['source'].map(source_mapping)

## Model 1: Baseline Model: Majority Class predictor:

Let us start off with a baseline model - which is a majority class predictor and check its accuracy:

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score

# Step 1: Split the dataset into train and test sets
X1 = train_df.drop(columns=['source'])  # Drop the target column for features
y1 = test_df['source']  # Target column
X_train1 = train_df['title_lemmatized']
y_train1 = train_df['source']
X_test1 = test_df['title_lemmatized']
y_test1 = test_df['source']

# Step 2: Find the majority class in the training set
majority_class = y_train1.mode()[0]

# Step 3: Predict the majority class for the test set
y_test_predictions = [majority_class] * len(y_test1)

# Step 4: Compare the predictions with the actual values in the test set
correct_predictions = y_test1 == y_test_predictions

# Step 5: Calculate accuracy
accuracy = correct_predictions.mean()  # This will give the percentage of correct predictions

print(f"Accuracy of Majority Class Predictor: {accuracy:.4f}")

Accuracy of Majority Class Predictor: 0.5553


## Model 2: Basic Logistic Regression with TF-IDF Features

Now, let us try out a basic logistic regression model with TF-IDF features and calculate accuracy:

In [ ]:
# Import necessary libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
# Step 1: Load data & Preprocess - already done

# Step 2: Prepare features (X) and labels (y)
# X = combined_data_usable_cleaned['title_lemmatized']  # Text data
# y = combined_data_usable_cleaned['source']  # Labels (FoxNews, NBC, etc.)

# Step 3: Split the data into training and testing sets (80% train, 20% test)
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# combine training and testing data
# train_df = pd.concat([X_train, y_train], axis=1)
# test_df = pd.concat([X_test, y_test], axis=1)

# save training and testing data into csv files
#train_df.to_csv('train_data.csv', index=False)
#test_df.to_csv('test_data.csv', index=False)

In [ ]:
# Step 4: Convert the labels to binary values (0 for 'FoxNews', 1 for 'NBC News')
# y_train = y_train.apply(lambda x: 1 if x == 'NBC News' else 0)
# y_test = y_test.apply(lambda x: 1 if x == 'NBC News' else 0)

# Step 5: Convert the text data to TF-IDF features
vectorizer = TfidfVectorizer(stop_words='english', max_features=100)
X_train_tfidf = vectorizer.fit_transform(X_train1)
X_test_tfidf = vectorizer.transform(X_test1)

# Step 6: Train a Logistic Regression model
model = LogisticRegression(max_iter=100)
model.fit(X_train_tfidf, y_train1)

# Step 7: Make predictions on the test set
y_pred = model.predict(X_test_tfidf)

# Step 8: Evaluate the model
# accuracy
accuracy = accuracy_score(y_test1, y_pred)
# f1 score
f1 = f1_score(y_test1, y_pred, average='weighted')
# precision
precision = precision_score(y_test1, y_pred, average='weighted')
# recall
recall = recall_score(y_test1, y_pred, average='weighted')
print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test F1 Score: {f1:.4f}")
print(f"Test Precision: {precision:.4f}")
print(f"Test Recall: {recall:.4f}")
print("Classification Report:\n", classification_report(y_test1, y_pred))

Test Accuracy: 0.6629
Test F1 Score: 0.6495
Test Precision: 0.6663
Test Recall: 0.6629
Classification Report:
               precision    recall  f1-score   support

           0       0.66      0.83      0.73       537
           1       0.68      0.46      0.55       430

    accuracy                           0.66       967
   macro avg       0.67      0.64      0.64       967
weighted avg       0.67      0.66      0.65       967



# Part 3: Additional Models

We will implement the following models. We will start by splitting our training data into a training and validation set before we begin with the models.
- LSTM
- Word embeddings with Word2Vec or glove
- RNN
- GRU
- CNN for text
- BERT

## Train/Val/Test Split

In [ ]:
# do train, val, test split on the data
from sklearn.model_selection import train_test_split
X = train_df['title_lemmatized']
y = train_df['source']
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

# make test set
X_test = test_df['title_lemmatized']
y_test = test_df['source']

# final data to use:
# X_train, y_train for training (60% of total data)
# X_val, y_val for validation (hyperparameter tuning) (20% of total data)
# X_test, y_test for testing (20% of total data)

# before tuning, use X and y for training

In [ ]:
pip install optuna

## Model 3: Word embeddings with a simple neural network:

### Model 3 Part 1: Standard Model

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import classification_report
from gensim.models import Word2Vec

# Step 0: Preprocess Input Data ---> why is this an issue??
def preprocess_sentences(sentences):
    # Replace NaN or non-string values with empty strings
    return [str(sentence) if isinstance(sentence, str) else "" for sentence in sentences]

# Preprocess X and X_test
X_train = preprocess_sentences(X_train)
X_val = preprocess_sentences(X_val)
X_test = preprocess_sentences(X_test)

# Step 1: Train Word2Vec Model
def train_word2vec(sentences, embedding_dim=100, min_count=1, window=5, sg=0):
    # Tokenized sentences for Word2Vec training
    tokenized_sentences = [sentence.split() for sentence in sentences]
    word2vec_model = Word2Vec(sentences=tokenized_sentences, vector_size=embedding_dim,
                              min_count=min_count, window=window, sg=sg)
    return word2vec_model

# Combine training and test data for Word2Vec training
all_sentences = X_train + X_val + X_test  # Assuming X and X_test are lists of text
embedding_dim = 100  # Dimension of word vectors
word2vec_model = train_word2vec(all_sentences, embedding_dim=embedding_dim)

# Step 2: Compute Average Embedding for Each Sentence
def compute_avg_word2vec(sentence, model, embedding_dim):
    # Tokenize the sentence and retrieve embeddings for known words
    words = sentence.split()
    word_embeddings = [model.wv[word] for word in words if word in model.wv]
    if word_embeddings:
        return np.mean(word_embeddings, axis=0)
    else:
        return np.zeros(embedding_dim)  # Fallback for sentences with no known words

X_train_embeddings = np.array([compute_avg_word2vec(sentence, word2vec_model, embedding_dim) for sentence in X_train])
X_val_embeddings = np.array([compute_avg_word2vec(sentence, word2vec_model, embedding_dim) for sentence in X_val])
X_test_embeddings = np.array([compute_avg_word2vec(sentence, word2vec_model, embedding_dim) for sentence in X_test])

# Step 3: Encode Labels
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)
y_test_encoded = label_encoder.transform(y_test)

from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import LeakyReLU

# Step 4: Define and Train Optimized Feedforward Neural Network
model = Sequential([
    Dense(256, activation='relu', input_shape=(embedding_dim,)),
    BatchNormalization(),
    Dropout(0.4),  # Increase dropout rate to reduce overfitting
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),
    Dense(len(np.unique(y_train_encoded)), activation='softmax')
])
'''
model = Sequential([
    Dense(256, input_shape=(embedding_dim,)),
    LeakyReLU(alpha=0.01),  # Use LeakyReLU with a small slope for negative values
    BatchNormalization(),
    Dropout(0.4),
    Dense(128),
    LeakyReLU(alpha=0.01),
    BatchNormalization(),
    Dropout(0.3),
    Dense(64),
    LeakyReLU(alpha=0.01),
    BatchNormalization(),
    Dropout(0.2),
    Dense(len(np.unique(y_train_encoded)), activation='softmax')
])
'''

# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Define callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)

# Train the model with callbacks
history = model.fit(
    X_train_embeddings, y_train_encoded,
    epochs=50,  # Increase epochs for more training
    batch_size=64,  # Try a larger batch size
    validation_split=0.2,
    callbacks=[early_stopping, reduce_lr]
)

# Step 5: Evaluate on Test Set
y_test_pred = model.predict(X_test_embeddings)
y_test_pred_classes = np.argmax(y_test_pred, axis=1)

# output evaluation metrics
# accuracy
accuracy = accuracy_score(y_test_encoded, y_test_pred_classes)
# f1 score
f1 = f1_score(y_test_encoded, y_test_pred_classes, average='weighted')
# precision
precision = precision_score(y_test_encoded, y_test_pred_classes, average='weighted')
# recall
recall = recall_score(y_test_encoded, y_test_pred_classes, average='weighted')
print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test F1 Score: {f1:.4f}")
print(f"Test Precision: {precision:.4f}")
print(f"Test Recall: {recall:.4f}")
print("Classification Report:\n", classification_report(y_test_encoded, y_test_pred_classes))

/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 12s 156ms/step - accuracy: 0.5274 - loss: 0.9424 - val_accuracy: 0.5241 - val_loss: 0.6928 - learning_rate: 0.0010
Epoch 2/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5323 - loss: 0.8047 - val_accuracy: 0.5241 - val_loss: 0.6973 - learning_rate: 0.0010
Epoch 3/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5156 - loss: 0.8198 - val_accuracy: 0.5241 - val_loss: 0.7011 - learning_rate: 0.0010
Epoch 4/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5643 - loss: 0.7470 - val_accuracy: 0.5241 - val_loss: 0.6962 - learning_rate: 0.0010
Epoch 5/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5345 - loss: 0.7516 - val_accuracy: 0.5241 - val_loss: 0.6954 - learning_rate: 5.0000e-04
Epoch 6/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5531 - loss: 0.7427 - val_accuracy: 0.5241 - val_loss: 0.6962 - learning_rate: 5.0000e-04
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step
Test Accuracy: 0.5553
Test F1 Score: 0.3966
Test 

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/m

### Model 3 Part 2: Hyperparameter Tuning

In [ ]:
pip install keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.1/129.1 kB 10.5 MB/s eta 0:00:00


In [ ]:
import keras_tuner as kt
from tensorflow.keras.layers import LeakyReLU

# Define the model building function for hyperparameter tuning
def build_model(hp):
    model = Sequential()

    # Hyperparameter for the number of units in the first layer
    model.add(Dense(
        units=hp.Int('units_1', min_value=64, max_value=512, step=64),
        activation='relu',
        input_shape=(embedding_dim,)
    ))

    model.add(BatchNormalization())
    model.add(Dropout(hp.Float('dropout_1', min_value=0.2, max_value=0.5, step=0.1)))

    # Hyperparameter for the number of units in the second layer
    model.add(Dense(
        units=hp.Int('units_2', min_value=64, max_value=512, step=64),
        activation='relu'
    ))
    model.add(BatchNormalization())
    model.add(Dropout(hp.Float('dropout_2', min_value=0.2, max_value=0.5, step=0.1)))

    # Hyperparameter for the number of units in the third layer
    model.add(Dense(
        units=hp.Int('units_3', min_value=64, max_value=512, step=64),
        activation='relu'
    ))
    model.add(BatchNormalization())
    model.add(Dropout(hp.Float('dropout_3', min_value=0.2, max_value=0.5, step=0.1)))

    # Output layer
    model.add(Dense(len(np.unique(y_train_encoded)), activation='softmax'))

    # Hyperparameter for learning rate
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=hp.Float('learning_rate', min_value=1e-5, max_value=1e-2, sampling='log')
        ),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [ ]:
# Instantiate the tuner
tuner = kt.Hyperband(
    build_model,
    objective='val_accuracy',  # We want to maximize validation accuracy
    max_epochs=10,  # Number of epochs to train each trial
    hyperband_iterations=2,  # Number of random search iterations
    directory='my_dir',
    project_name='hyperparameter_tuning'
)

# Define early stopping
stop_early = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5)

# Perform the search
tuner.search(X_val_embeddings, y_val_encoded, epochs=50, validation_data=(X_val_embeddings, y_val_encoded), callbacks=[EarlyStopping(monitor='val_loss', patience=3)])

# Get the best model and hyperparameters
best_model = tuner.get_best_models(num_models=1)[0]
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]

print("Best Hyperparameters: ", best_hp.values)

Trial 60 Complete [00h 00m 13s]
val_accuracy: 0.4668737053871155

Best val_accuracy So Far: 0.6438923478126526
Total elapsed time: 00h 15m 35s
Best Hyperparameters:  {'units_1': 512, 'dropout_1': 0.2, 'units_2': 448, 'dropout_2': 0.2, 'units_3': 448, 'dropout_3': 0.30000000000000004, 'learning_rate': 0.008640941672309294, 'tuner/epochs': 10, 'tuner/initial_epoch': 4, 'tuner/bracket': 1, 'tuner/round': 1, 'tuner/trial_id': '0050'}


In [ ]:
# Evaluate the best model on the test set
y_test_pred = best_model.predict(X_test_embeddings)
y_test_pred_classes = np.argmax(y_test_pred, axis=1)

accuracy = accuracy_score(y_test_encoded, y_test_pred_classes)
print(f"Test Accuracy: {accuracy:.4f}")
print("Classification Report:\n", classification_report(y_test_encoded, y_test_pred_classes))

31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step
Test Accuracy: 0.6143
Classification Report:
               precision    recall  f1-score   support

           0       0.60      0.90      0.72       537
           1       0.68      0.25      0.37       430

    accuracy                           0.61       967
   macro avg       0.64      0.58      0.55       967
weighted avg       0.64      0.61      0.57       967



## Model 4: Recurrent neural network (RNNs)

### Model 4 Part 1: Standard Model

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Embedding, Dropout, GRU
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Step 0: Preprocess Input Data (Assuming X and X_test are lists of titles)
def preprocess_sentences(sentences):
    # Replace NaN or non-string values with empty strings
    return [str(sentence) if isinstance(sentence, str) else "" for sentence in sentences]

X_train = preprocess_sentences(X_train)  # Your training titles
X_val = preprocess_sentences(X_val)  # Your validation titles
X_test = preprocess_sentences(X_test)  # Your testing titles

# Step 1: Tokenize the text data (titles)
tokenizer = Tokenizer()
tokenizer.fit_on_texts(X_train)

# Convert texts to sequences of integers
X_train_sequences = tokenizer.texts_to_sequences(X_train)
X_val_sequences = tokenizer.texts_to_sequences(X_val)
X_test_sequences = tokenizer.texts_to_sequences(X_test)

# Step 2: Pad sequences to ensure uniform input size
max_sequence_length = 100  # Choose a suitable length based on your data
X_train_padded = pad_sequences(X_train_sequences, maxlen=max_sequence_length)
X_val_padded = pad_sequences(X_val_sequences, maxlen=max_sequence_length)
X_test_padded = pad_sequences(X_test_sequences, maxlen=max_sequence_length)

# Step 3: Encode Labels
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)
y_test_encoded = label_encoder.transform(y_test)

# Step 4: Build RNN or LSTM Model
model = Sequential([
    # Embedding Layer
    Embedding(input_dim=len(tokenizer.word_index) + 1,  # Size of vocabulary
              output_dim=100,  # Dimension of word vectors
              input_length=max_sequence_length),

    # LSTM Layer (we can replace with GRU if we want to try that)
    LSTM(128, return_sequences=False, dropout=0.2, recurrent_dropout=0.2),  # LSTM with dropout

    # Dense Layer
    Dense(64, activation='relu'),
    Dropout(0.3),

    # Output Layer
    Dense(len(np.unique(y_train_encoded)), activation='softmax')
])

# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Define callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)

# Step 5: Train the model
history = model.fit(
    X_train_padded, y_train_encoded,
    epochs=20,  # You can adjust epochs depending on your data
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stopping, reduce_lr]
)

# Step 6: Evaluate on Test Set
y_test_pred = model.predict(X_test_padded)
y_test_pred_classes = np.argmax(y_test_pred, axis=1)

accuracy = accuracy_score(y_test_encoded, y_test_pred_classes)
print(f"Test Accuracy: {accuracy:.4f}")
print("Classification Report:\n", classification_report(y_test_encoded, y_test_pred_classes))

Epoch 1/20


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


37/37 ━━━━━━━━━━━━━━━━━━━━ 16s 208ms/step - accuracy: 0.5352 - loss: 0.6883 - val_accuracy: 0.6724 - val_loss: 0.6423 - learning_rate: 0.0010
Epoch 2/20
37/37 ━━━━━━━━━━━━━━━━━━━━ 16s 251ms/step - accuracy: 0.7305 - loss: 0.5687 - val_accuracy: 0.6672 - val_loss: 0.5941 - learning_rate: 0.0010
Epoch 3/20
37/37 ━━━━━━━━━━━━━━━━━━━━ 9s 220ms/step - accuracy: 0.8922 - loss: 0.2608 - val_accuracy: 0.7448 - val_loss: 0.6307 - learning_rate: 0.0010
Epoch 4/20
37/37 ━━━━━━━━━━━━━━━━━━━━ 8s 227ms/step - accuracy: 0.9712 - loss: 0.0901 - val_accuracy: 0.7517 - val_loss: 0.8206 - learning_rate: 0.0010
Epoch 5/20
37/37 ━━━━━━━━━━━━━━━━━━━━ 10s 223ms/step - accuracy: 0.9934 - loss: 0.0346 - val_accuracy: 0.7259 - val_loss: 0.9521 - learning_rate: 0.0010
Epoch 6/20
37/37 ━━━━━━━━━━━━━━━━━━━━ 11s 208ms/step - accuracy: 0.9936 - loss: 0.0246 - val_accuracy: 0.7310 - val_loss: 1.1641 - learning_rate: 5.0000e-04
Epoch 7/20
37/37 ━━━━━━━━━━━━━━━━━━━━ 11s 251ms/step - accuracy: 0.9960 - loss: 0.0096 - va

In [ ]:
import keras_tuner as kt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Embedding, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.metrics import accuracy_score
import numpy as np

# Preprocess the data as before
def preprocess_sentences(sentences):
    return [str(sentence) if isinstance(sentence, str) else "" for sentence in sentences]

X_train = preprocess_sentences(X_train)  # Your training titles
X_val = preprocess_sentences(X_val)  # Your validation titles
X_test = preprocess_sentences(X_test)  # Your testing titles

# Tokenize the text data
tokenizer = Tokenizer()
tokenizer.fit_on_texts(X_train)

X_train_sequences = tokenizer.texts_to_sequences(X_train)
X_val_sequences = tokenizer.texts_to_sequences(X_val)
X_test_sequences = tokenizer.texts_to_sequences(X_test)

# Pad sequences
max_sequence_length = 100
X_train_padded = pad_sequences(X_train_sequences, maxlen=max_sequence_length)
X_val_padded = pad_sequences(X_val_sequences, maxlen=max_sequence_length)
X_test_padded = pad_sequences(X_test_sequences, maxlen=max_sequence_length)

# Encode Labels
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)
y_test_encoded = label_encoder.transform(y_test)

### Model 4 Part 2: Hyperparameter Tuning

In [ ]:
## Hyperparameter tuning
from kerastuner import HyperModel, RandomSearch
from tensorflow.keras import layers
import tensorflow as tf
from tensorflow.keras.models import Sequential

# Define the HyperModel for Keras Tuner
class MyHyperModel(HyperModel):
    def build(self, hp):
        model = Sequential()
        # Embedding Layer
        model.add(Embedding(input_dim=len(tokenizer.word_index) + 1,  # Size of vocabulary
                            output_dim=100,  # Dimension of word vectors
                            input_length=max_sequence_length))

        # LSTM Layer (hyperparameter tuning for number of units)
        model.add(LSTM(units=hp.Int('lstm_units', min_value=64, max_value=256, step=64),
                       return_sequences=False,
                       dropout=hp.Float('dropout_rate', min_value=0.2, max_value=0.5, step=0.1),
                       recurrent_dropout=0.2))

        # Dense Layer
        model.add(Dense(units=hp.Int('dense_units', min_value=64, max_value=256, step=64),
                        activation='relu'))
        model.add(Dropout(0.3))

        # Output Layer
        model.add(Dense(len(np.unique(y_train_encoded)), activation='softmax'))

        # Compile the model
        model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=hp.Float('learning_rate', min_value=1e-5, max_value=1e-2, sampling='LOG')),
                      loss='sparse_categorical_crossentropy', metrics=['accuracy'])
        return model

# Initialize Keras Tuner RandomSearch
tuner = RandomSearch(
    MyHyperModel(),
    objective='val_accuracy',
    max_trials=30,  # Set the number of trials you want to run (e.g., 30)
    executions_per_trial=1,
    directory='hyperparameter_tuning',
    project_name='text_classification'
)

# Step 1: Fit the tuner
tuner.search(X_train_padded, y_train_encoded, epochs=10, batch_size=64, validation_data=(X_val_padded, y_val_encoded),
             callbacks=[EarlyStopping(monitor='val_loss', patience=3)])

# Step 2: Get the best model and hyperparameters
best_model = tuner.get_best_models(num_models=1)[0]
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]

# Step 3: Evaluate the best model on test data
y_test_pred = best_model.predict(X_test_padded)
y_test_pred_classes = np.argmax(y_test_pred, axis=1)

accuracy = accuracy_score(y_test_encoded, y_test_pred_classes)
print(f"Test Accuracy with best hyperparameters: {accuracy:.4f}")

# print best parameters for saving
print("Best Hyperparameters: ", best_hp.values)

Trial 30 Complete [00h 00m 59s]
val_accuracy: 0.7650103569030762

Best val_accuracy So Far: 0.7712215185165405
Total elapsed time: 00h 55m 47s


/usr/local/lib/python3.10/dist-packages/keras/src/saving/saving_lib.py:713: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


31/31 ━━━━━━━━━━━━━━━━━━━━ 2s 49ms/step
Test Accuracy with best hyperparameters: 0.7787
Best Hyperparameters:  {'lstm_units': 192, 'dropout_rate': 0.4, 'dense_units': 192, 'learning_rate': 0.0002717199130284055}


## Model 5: Convolutional Neural Network (CNN) for Text
How it works:
- Apply convolution filters to capture n-gram features.

Steps:
- Tokenize and pad sequences.
- Use an embedding layer to represent words.
- Pass sequences through convolutional and pooling layers.
- Use a dense layer for classification.
- Pros: Captures local patterns (e.g., phrases) in text efficiently.
- Cons: Limited in capturing long-range dependencies.

In [ ]:
mean_words = train_df['title_lemmatized'].apply(lambda x: len(str(x).split())).mean()

print(f"The mean number of words in X_train is: {mean_words}")

The mean number of words in X_train is: 9.616977225672878


### Model 5 Part 1: Standard Model

In [ ]:
MAX_VOCAB_SIZE = 5000  # Headlines: good vocab size is 5000
MAX_SEQUENCE_LENGTH = 10  # Avg length of the headline input: 9.6, so using 10
EMBEDDING_DIM = 100  # Simplicity vs power: need to tune maybe?

# Tokenize and pad sequences
tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE)
tokenizer.fit_on_texts(X)

# Texts to sequences
X_train_seq = tokenizer.texts_to_sequences(X)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# Pad sequences
X_train_padded = pad_sequences(X_train_seq, maxlen=MAX_SEQUENCE_LENGTH, padding='post')
X_test_padded = pad_sequences(X_test_seq, maxlen=MAX_SEQUENCE_LENGTH, padding='post')

# CNN model
# With ReLU activation
# 128 filters, kernel size is 3
# Sigmoid for binary classification
model = Sequential([
    Embedding(input_dim=MAX_VOCAB_SIZE, output_dim=EMBEDDING_DIM, input_length=MAX_SEQUENCE_LENGTH),
    Conv1D(filters=128, kernel_size=3, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

# Compile and train
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


# Evaluate
loss, accuracy = model.evaluate(X_test_padded, y_test)
print(f'Test Loss: {loss}')
print(f'Test Accuracy: {accuracy}')

/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


31/31 ━━━━━━━━━━━━━━━━━━━━ 3s 32ms/step - accuracy: 0.5755 - loss: 0.6883
Test Loss: 0.6894015073776245
Test Accuracy: 0.5553257465362549


### Model 5 Part 2: Hyperparameter Tuning


In [ ]:
import optuna

def objective(trial):
    # define hyperparameter ranges
    MAX_VOCAB_SIZE = trial.suggest_int('max_vocab_size', 500, 50000)
    MAX_SEQUENCE_LENGTH = trial.suggest_int('max_sequence_length', 3, 20)
    EMBEDDING_DIM = trial.suggest_categorical('embedding_dim', [32, 64, 100, 300, 512, 768, 1024])

    # Tokenize and pad sequences
    tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE)
    tokenizer.fit_on_texts(X_train)

    # Texts to sequences
    X_train_seq = tokenizer.texts_to_sequences(X_train)
    X_val_seq = tokenizer.texts_to_sequences(X_val)

    # Pad sequences
    X_train_padded = pad_sequences(X_train_seq, maxlen=MAX_SEQUENCE_LENGTH, padding='post')
    X_val_padded = pad_sequences(X_val_seq, maxlen=MAX_SEQUENCE_LENGTH, padding='post')

    # CNN model
    # With ReLU activation
    # 128 filters, kernel size is 3
    # Sigmoid for binary classification
    model = Sequential([
        Embedding(input_dim=MAX_VOCAB_SIZE, output_dim=EMBEDDING_DIM, input_length=MAX_SEQUENCE_LENGTH),
        Conv1D(filters=128, kernel_size=3, activation='relu'),
        GlobalMaxPooling1D(),
        Dense(64, activation='relu'),
        Dense(1, activation='sigmoid')
    ])

    # Compile and train
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    history = model.fit(X_train_padded, y_train, epochs=20, batch_size=32, validation_data=(X_val_padded, y_val))

    # Evaluate
    loss, accuracy = model.evaluate(X_val_padded, y_val)

    print(f'Test Loss: {loss}')
    print(f'Test Accuracy: {accuracy}')

    # return the validation accuracy as the metric to maximize
    return accuracy

In [ ]:
# create an optuna study and optimize
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)

# display best hyperparameters
print("Best trial:", study.best_trial.params)

# save the best parameters
best_params = study.best_trial.params

[I 2024-12-14 02:05:10,335] A new study created in memory with name: no-name-9d7b4234-1ecb-4e3d-9af8-08c0379b2ba2


Epoch 1/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 7s 33ms/step - accuracy: 0.6036 - loss: 0.6639 - val_accuracy: 0.6957 - val_loss: 0.5704
Epoch 2/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8962 - loss: 0.3151 - val_accuracy: 0.7215 - val_loss: 0.6069
Epoch 3/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9866 - loss: 0.0573 - val_accuracy: 0.7360 - val_loss: 0.7480
Epoch 4/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9992 - loss: 0.0084 - val_accuracy: 0.7288 - val_loss: 0.8879
Epoch 5/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9990 - loss: 0.0036 - val_accuracy: 0.7371 - val_loss: 0.9220
Epoch 6/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9992 - loss: 0.0037 - val_accuracy: 0.7360 - val_loss: 0.9990
Epoch 7/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9989 - loss: 0.0020 - val_accuracy: 0.7402 - val_loss: 1.0154
Epoch 8/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 1.0000 - loss: 8.5512e-04 - val_accuracy: 0.7371 - val_

[I 2024-12-14 02:05:29,710] Trial 0 finished with value: 0.7360248565673828 and parameters: {'max_vocab_size': 16275, 'max_sequence_length': 9, 'embedding_dim': 300}. Best is trial 0 with value: 0.7360248565673828.


Test Loss: 1.3508868217468262
Test Accuracy: 0.7360248565673828
Epoch 1/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 4s 19ms/step - accuracy: 0.5600 - loss: 0.6804 - val_accuracy: 0.7091 - val_loss: 0.6032
Epoch 2/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8809 - loss: 0.3909 - val_accuracy: 0.7464 - val_loss: 0.5376
Epoch 3/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9747 - loss: 0.1040 - val_accuracy: 0.7505 - val_loss: 0.7247
Epoch 4/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9967 - loss: 0.0214 - val_accuracy: 0.7381 - val_loss: 0.8388
Epoch 5/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9990 - loss: 0.0067 - val_accuracy: 0.7474 - val_loss: 0.9224
Epoch 6/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9989 - loss: 0.0045 - val_accuracy: 0.7319 - val_loss: 1.0152
Epoch 7/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9991 - loss: 0.0031 - val_accuracy: 0.7319 - val_loss: 1.0457
Epoch 8/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - a

[I 2024-12-14 02:05:42,287] Trial 1 finished with value: 0.7370600700378418 and parameters: {'max_vocab_size': 35685, 'max_sequence_length': 15, 'embedding_dim': 64}. Best is trial 1 with value: 0.7370600700378418.


...

Test Loss: 1.2710169553756714
Test Accuracy: 0.7577639818191528
Epoch 1/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 4s 19ms/step - accuracy: 0.5678 - loss: 0.6790 - val_accuracy: 0.5890 - val_loss: 0.6465
Epoch 2/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8075 - loss: 0.4402 - val_accuracy: 0.7391 - val_loss: 0.5333
Epoch 3/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9495 - loss: 0.1593 - val_accuracy: 0.7412 - val_loss: 0.6855
Epoch 4/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9941 - loss: 0.0359 - val_accuracy: 0.7153 - val_loss: 0.8889
Epoch 5/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9980 - loss: 0.0132 - val_accuracy: 0.7226 - val_loss: 0.9870
Epoch 6/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9989 - loss: 0.0069 - val_accuracy: 0.7143 - val_loss: 1.0603
Epoch 7/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9973 - loss: 0.0055 - val_accuracy: 0.7184 - val_loss: 1.1191
Epoch 8/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - a

[I 2024-12-14 02:07:47,056] Trial 8 finished with value: 0.7142857313156128 and parameters: {'max_vocab_size': 17804, 'max_sequence_length': 13, 'embedding_dim': 32}. Best is trial 2 with value: 0.7629399299621582.


Test Loss: 1.6929930448532104
Test Accuracy: 0.7142857313156128
Epoch 1/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step - accuracy: 0.5577 - loss: 0.6862 - val_accuracy: 0.6625 - val_loss: 0.6597
Epoch 2/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8246 - loss: 0.4864 - val_accuracy: 0.7039 - val_loss: 0.6120
Epoch 3/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9528 - loss: 0.1549 - val_accuracy: 0.6894 - val_loss: 0.8250
Epoch 4/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9924 - loss: 0.0458 - val_accuracy: 0.6905 - val_loss: 0.9937
Epoch 5/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9968 - loss: 0.0162 - val_accuracy: 0.6957 - val_loss: 1.1331
Epoch 6/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9964 - loss: 0.0094 - val_accuracy: 0.6977 - val_loss: 1.2348
Epoch 7/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9993 - loss: 0.0040 - val_accuracy: 0.6967 - val_loss: 1.2945
Epoch 8/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - a

[I 2024-12-14 02:07:57,232] Trial 9 finished with value: 0.6977225542068481 and parameters: {'max_vocab_size': 21949, 'max_sequence_length': 7, 'embedding_dim': 32}. Best is trial 2 with value: 0.7629399299621582.


Test Loss: 1.7428081035614014
Test Accuracy: 0.6977225542068481
Best trial: {'max_vocab_size': 22867, 'max_sequence_length': 12, 'embedding_dim': 512}


In [ ]:
# Tokenize and pad sequences
tokenizer = Tokenizer(num_words=best_params['max_vocab_size'])
tokenizer.fit_on_texts(X_train)

# Texts to sequences
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# Pad sequences
X_train_padded = pad_sequences(X_train_seq, maxlen=best_params['max_sequence_length'], padding='post')
X_test_padded = pad_sequences(X_test_seq, maxlen=best_params['max_sequence_length'], padding='post')

# CNN model
# With ReLU activation
# 128 filters, kernel size is 3
# Sigmoid for binary classification
model = Sequential([
    Embedding(input_dim=best_params['max_vocab_size'], output_dim=best_params['embedding_dim'], input_length=best_params['max_sequence_length']),
    Conv1D(filters=128, kernel_size=3, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

# Compile and train
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
history = model.fit(X_train_padded, y_train, epochs=20, batch_size=32)

# Evaluate
loss, accuracy = model.evaluate(X_test_padded, y_test)
print(f'Test Loss: {loss}')
print(f'Test Accuracy: {accuracy}')
# print classificatio report
y_pred = (model.predict(X_test_padded) > 0.5).astype("int32")
print(classification_report(y_test, y_pred))

Epoch 1/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - accuracy: 0.6109 - loss: 0.6500
Epoch 2/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9052 - loss: 0.2525
Epoch 3/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9906 - loss: 0.0478
Epoch 4/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9990 - loss: 0.0064
Epoch 5/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9985 - loss: 0.0049
Epoch 6/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9986 - loss: 0.0028
Epoch 7/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9985 - loss: 0.0024
Epoch 8/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9995 - loss: 0.0020
Epoch 9/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9987 - loss: 0.0024
Epoch 10/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9997 - loss: 0.0013
Epoch 11/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9990 - loss: 0.0017
Epoch 12/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9999 - 

## Model 6: Transformer-based model: BERT

### Model 6 Part 1: Standard Model

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import Trainer, TrainingArguments
from transformers import EarlyStoppingCallback
import os
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Disable W&B logging
os.environ["WANDB_DISABLED"] = "true"

# Load tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased',
                                          hidden_dropout_prob=0.3,
                                          attention_probs_dropout_prob=0.3)  # Combat overfitting
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# Tokenize data
train_encodings = tokenizer(list(X), truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(list(X_test), truncation=True, padding=True, max_length=128)

# Create Dataset class
class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

# Create train and test datasets
train_dataset = Dataset(train_encodings, list(y))
test_dataset = Dataset(test_encodings, list(y_test))

# Define training arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    weight_decay=0.00033497624679702326,
    logging_dir='./logs',
    logging_steps=10,
    evaluation_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    learning_rate=2.6752066947201052e-05  # Value found from optuna tuning
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],  # Early stopping
)

# Train the model
trainer.train()

# Evaluate the model
eval_results = trainer.evaluate(test_dataset)
print(f"Evaluation Results: {eval_results}")

# Predict and evaluate metrics
predictions = trainer.predict(test_dataset)
preds = predictions.predictions.argmax(axis=1)  # Predicted labels
labels = predictions.label_ids  # True labels

# Calculate accuracy
accuracy = accuracy_score(labels, preds)
print(f"Accuracy: {accuracy}")

# Print classification report
class_report = classification_report(labels, preds, target_names=["Class 0", "Class 1"])  # Update target names as needed
print("Classification Report:")
print(class_report)

# Print confusion matrix
conf_matrix = confusion_matrix(labels, preds)
print("Confusion Matrix:")
print(conf_matrix)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Step,Training Loss,Validation Loss
500,0.156800,0.621956


Evaluation Results: {'eval_loss': 0.6219556927680969, 'eval_runtime': 1.8082, 'eval_samples_per_second': 534.776, 'eval_steps_per_second': 17.144, 'epoch': 5.0}
Accuracy: 0.7745604963805585
Classification Report:
              precision    recall  f1-score   support

     Class 0       0.83      0.75      0.79       537
     Class 1       0.72      0.81      0.76       430

    accuracy                           0.77       967
   macro avg       0.77      0.78      0.77       967
weighted avg       0.78      0.77      0.78       967

Confusion Matrix:
[[402 135]
 [ 83 347]]


### Model 5 Part 2: Hyperparameter Tuning

In [ ]:
# hyperparameter tuning

from transformers import Trainer, TrainingArguments
from transformers import BertTokenizer, BertForSequenceClassification
import os
import torch
from sklearn.metrics import accuracy_score
from transformers import EarlyStoppingCallback
from transformers import get_cosine_with_hard_restarts_schedule_with_warmup

# disable WandB
os.environ["WANDB_DISABLED"] = "true"

# define Dataset class
class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

# define tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2, hidden_dropout_prob=0.3)

train_encodings = tokenizer(list(X_train), truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(list(X_val), truncation=True, padding=True, max_length=128)

train_dataset = Dataset(train_encodings, list(y_train))
val_dataset = Dataset(val_encodings, list(y_val))

def objective(trial):
    # define hyperparameter ranges
    learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-4)
    num_train_epochs = trial.suggest_int('num_train_epochs', 3, 5)
    train_batch_size = trial.suggest_categorical('per_device_train_batch_size', [16, 32, 64])
    eval_batch_size = trial.suggest_categorical('per_device_eval_batch_size', [16, 32, 64])
    weight_decay = trial.suggest_loguniform('weight_decay', 1e-4, 1e-3)

    # define early stopping callback
    early_stopping = EarlyStoppingCallback(
        early_stopping_patience=3,  # number of epochs to wait for improvement
        early_stopping_threshold=0.01  # minimum change to qualify as improvement
    )

    # use adam for l2 regularization to combat overfitting
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)


    # define training arguments with learning rate scheduling
    training_args = TrainingArguments(
        output_dir='./results',
        num_train_epochs=num_train_epochs,
        per_device_train_batch_size=train_batch_size,
        per_device_eval_batch_size=eval_batch_size,
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        evaluation_strategy="steps",
        greater_is_better=False,
        load_best_model_at_end=True
    )

    # initialize Trainer with learning rate scheduler
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,  # Training set
        eval_dataset=val_dataset,     # Validation set
        callbacks=[early_stopping]
    )

    # train and evaluate
    trainer.train()
    predictions, labels, _ = trainer.predict(val_dataset)
    predictions = torch.argmax(torch.tensor(predictions), axis=1)
    accuracy = accuracy_score(labels, predictions)

    # return the validation accuracy as the metric to maximize
    return accuracy


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# create an optuna study and optimize
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)

# display best hyperparameters
print("Best trial:", study.best_trial.params)

# save the best parameters
best_params = study.best_trial.params

[I 2024-12-14 02:28:56,728] A new study created in memory with name: no-name-5532bf1f-3367-427a-98f6-930bbee5eb43
<ipython-input-24-b7bd0851a59a>:40: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-4)
<ipython-input-24-b7bd0851a59a>:44: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  weight_decay = trial.suggest_loguniform('weight_decay', 1e-4, 1e-3)
/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Using the `WANDB_DISABLED` environm

Step,Training Loss,Validation Loss


[I 2024-12-14 02:29:57,353] Trial 0 finished with value: 0.7567287784679089 and parameters: {'learning_rate': 7.545969011015214e-05, 'num_train_epochs': 3, 'per_device_train_batch_size': 32, 'per_device_eval_batch_size': 16, 'weight_decay': 0.0009563874807305245}. Best is trial 0 with value: 0.7567287784679089.
<ipython-input-24-b7bd0851a59a>:40: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-4)
<ipython-input-24-b7bd0851a59a>:44: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  weight_decay = trial.suggest_loguniform('weight_decay', 1e-4, 1e-3)
/usr/local/lib/python3.10/dist-packages/transformers/training_a

Step,Training Loss,Validation Loss
500,0.382500,0.619856


[I 2024-12-14 02:33:15,921] Trial 1 finished with value: 0.7391304347826086 and parameters: {'learning_rate': 3.227620954227307e-05, 'num_train_epochs': 3, 'per_device_train_batch_size': 16, 'per_device_eval_batch_size': 32, 'weight_decay': 0.0007012479244663641}. Best is trial 0 with value: 0.7567287784679089.
<ipython-input-24-b7bd0851a59a>:40: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-4)
<ipython-input-24-b7bd0851a59a>:44: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  weight_decay = trial.suggest_loguniform('weight_decay', 1e-4, 1e-3)
/usr/local/lib/python3.10/dist-packages/transformers/training_a

...

Step,Training Loss,Validation Loss
500,0.060700,2.265810


[I 2024-12-14 02:51:17,235] Trial 8 finished with value: 0.727743271221532 and parameters: {'learning_rate': 1.1040986523296721e-05, 'num_train_epochs': 5, 'per_device_train_batch_size': 16, 'per_device_eval_batch_size': 16, 'weight_decay': 0.00014087129864506897}. Best is trial 0 with value: 0.7567287784679089.
<ipython-input-24-b7bd0851a59a>:40: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-4)
<ipython-input-24-b7bd0851a59a>:44: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  weight_decay = trial.suggest_loguniform('weight_decay', 1e-4, 1e-3)
/usr/local/lib/python3.10/dist-packages/transformers/training_

Step,Training Loss,Validation Loss


[I 2024-12-14 02:53:17,945] Trial 9 finished with value: 0.722567287784679 and parameters: {'learning_rate': 4.357341793153824e-05, 'num_train_epochs': 5, 'per_device_train_batch_size': 64, 'per_device_eval_batch_size': 16, 'weight_decay': 0.00020103123479903198}. Best is trial 0 with value: 0.7567287784679089.


Best trial: {'learning_rate': 7.545969011015214e-05, 'num_train_epochs': 3, 'per_device_train_batch_size': 32, 'per_device_eval_batch_size': 16, 'weight_decay': 0.0009563874807305245}


In [ ]:
# after tuning, evaluate on the test set

test_encodings = tokenizer(list(X_test), truncation=True, padding=True, max_length=128)
test_dataset = Dataset(test_encodings, list(y_test))

# train final model using best parameters
final_training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=best_params['num_train_epochs'],
    per_device_train_batch_size=best_params['per_device_train_batch_size'],
    per_device_eval_batch_size=best_params['per_device_eval_batch_size'],
    learning_rate=best_params['learning_rate'],
    weight_decay=best_params['weight_decay'],
    evaluation_strategy="epoch"
)

final_trainer = Trainer(
    model=model,
    args=final_training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)
final_trainer.train()

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Epoch,Training Loss,Validation Loss
1,No log,1.686873
2,No log,1.499533
3,No log,1.982772


TrainOutput(global_step=273, training_loss=0.05186970679314582, metrics={'train_runtime': 138.0718, 'train_samples_per_second': 62.967, 'train_steps_per_second': 1.977, 'total_flos': 116161475386320.0, 'train_loss': 0.05186970679314582, 'epoch': 3.0})

In [ ]:
# post-tuning accuracy
# evaluate final model on the test set

predictions = final_trainer.predict(test_dataset)
logits = predictions.predictions
y_pred_bert = np.argmax(logits, axis=1)

# Evaluate
print(f"Accuracy: {accuracy_score(y_test, y_pred_bert):.4f}")
print("Classification Report:\n", classification_report(y_test, y_pred_bert))

Accuracy: 0.7342
Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.67      0.74       537
           1       0.67      0.81      0.73       430

    accuracy                           0.73       967
   macro avg       0.74      0.74      0.73       967
weighted avg       0.75      0.73      0.73       967



# Part 4: Ensemble Methods: CNN, LSTM and BERT:

We will now implement ensemble methods on our best models. Since our top 3 performers were the CNN, LSTM, and BERT models, we will proceed with bagging and boosting for these.

For bagging, we will implement a RandomForest model. For boosting, we will implement XGBoost.

In [ ]:
pip install keras

In [ ]:
pip install tensorflow

In [ ]:
pip uninstall tensorflow

Found existing installation: tensorflow 2.17.1
Uninstalling tensorflow-2.17.1:
  Would remove:
    /usr/local/bin/import_pb_to_tensorboard
    /usr/local/bin/saved_model_cli
    /usr/local/bin/tensorboard
    /usr/local/bin/tf_upgrade_v2
    /usr/local/bin/tflite_convert
    /usr/local/bin/toco
    /usr/local/bin/toco_from_protos
    /usr/local/lib/python3.10/dist-packages/tensorflow-2.17.1.dist-info/*
    /usr/local/lib/python3.10/dist-packages/tensorflow/*
Proceed (Y/n)? Y
  Successfully uninstalled tensorflow-2.17.1


In [ ]:
pip install tensorflow==2.10

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 578.0/578.0 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.7/438.7 kB 31.3 MB/s eta 0:00:00
  Attempting uninstall: keras
    Found existing installation: keras 2.8.0
    Uninstalling keras-2.8.0:
      Successfully uninstalled keras-2.8.0
  Attempting uninstall: protobuf
    Found existing installation: protobuf 4.25.5
    Uninstalling protobuf-4.25.5:
      Successfully uninstalled protobuf-4.25.5
  Attempting uninstall: gast
    Found existing installation: gast 0.6.0
    Uninstalling gast-0.6.0:
      Successfully uninstalled gast-0.6.0
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.8.0
    Uninstalling tensorboard-2.8.0:
      Successfully uninstalled

## Best Model: CNN

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Embedding, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.metrics import accuracy_score

In [ ]:
best_params_cnn = {'max_vocab_size': 22867, 'max_sequence_length': 12, 'embedding_dim': 512}
from tensorflow.keras.layers import Conv1D, GlobalMaxPooling1D

# Tokenize and pad sequences
tokenizer = Tokenizer(num_words=best_params_cnn['max_vocab_size'])
tokenizer.fit_on_texts(X_train)

# Texts to sequences
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# Pad sequences
X_train_padded = pad_sequences(X_train_seq, maxlen=best_params_cnn['max_sequence_length'], padding='post')
X_test_padded = pad_sequences(X_test_seq, maxlen=best_params_cnn['max_sequence_length'], padding='post')

# CNN model
# With ReLU activation
# 128 filters, kernel size is 3
# Sigmoid for binary classification
cnn_model = Sequential([
    Embedding(input_dim=best_params_cnn['max_vocab_size'], output_dim=best_params_cnn['embedding_dim'], input_length=best_params_cnn['max_sequence_length']),
    Conv1D(filters=128, kernel_size=3, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

cnn_model.build(input_shape=(None, best_params_cnn['max_sequence_length']))
dummy_data = np.zeros((1, best_params_cnn['max_sequence_length']), dtype=np.float32)
_ = cnn_model(dummy_data)  # Call the model


In [ ]:
# Compile and train
cnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cnn_history = cnn_model.fit(X_train_padded, y_train, epochs=20, batch_size=32)

# Evaluate
cnn_loss, cnn_accuracy = cnn_model.evaluate(X_test_padded, y_test)
print(f'CNN Test Loss: {cnn_loss}')
print(f'Test CNN Accuracy: {cnn_accuracy}')
# print classificatio report
y_pred_cnn = (cnn_model.predict(X_test_padded) > 0.5).astype("int32")

Epoch 1/20
91/91 [==============================] - 28s 297ms/step - loss: 0.5969 - accuracy: 0.6725
Epoch 2/20
91/91 [==============================] - 26s 289ms/step - loss: 0.2134 - accuracy: 0.9224
Epoch 3/20
91/91 [==============================] - 24s 263ms/step - loss: 0.0291 - accuracy: 0.9938
Epoch 4/20
91/91 [==============================] - 27s 299ms/step - loss: 0.0062 - accuracy: 0.9983
Epoch 5/20
91/91 [==============================] - 27s 294ms/step - loss: 0.0032 - accuracy: 0.9990
Epoch 6/20
91/91 [==============================] - 27s 302ms/step - loss: 0.0022 - accuracy: 0.9990
Epoch 7/20
91/91 [==============================] - 27s 296ms/step - loss: 0.0024 - accuracy: 0.9990
Epoch 8/20
91/91 [==============================] - 27s 292ms/step - loss: 0.0021 - accuracy: 0.9990
Epoch 9/20
91/91 [==============================] - 27s 292ms/step - loss: 0.0014 - accuracy: 0.9993
Epoch 10/20
91/91 [==============================] - 27s 293ms/step - loss: 0.0015 - accura

NameError: name 'classification_report' is not defined

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_cnn))

              precision    recall  f1-score   support

           0       0.80      0.83      0.81       537
           1       0.77      0.74      0.76       430

    accuracy                           0.79       967
   macro avg       0.79      0.78      0.79       967
weighted avg       0.79      0.79      0.79       967



In [ ]:
cnn_model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_2 (Embedding)     (None, 12, 512)           11707904  
                                                                 
 conv1d_1 (Conv1D)           (None, 10, 128)           196736    
                                                                 
 global_max_pooling1d (Globa  (None, 128)              0         
 lMaxPooling1D)                                                  
                                                                 
 dense (Dense)               (None, 64)                8256      
                                                                 
 dense_1 (Dense)             (None, 1)                 65        
                                                                 
Total params: 11,912,961
Trainable params: 11,912,961
Non-trainable params: 0
____________________________________________

In [ ]:
from tensorflow.keras.models import Model
import numpy as np

In [ ]:
# Access the layer by index (2 is for GlobalMaxPooling1D in your model)
global_max_layer = cnn_model.get_layer('global_max_pooling1d')

# Build the feature extractor
feature_extractor = Model(inputs=cnn_model.input, outputs=global_max_layer.output)

# Extract features for train and test data
X_train_features_cnn = feature_extractor.predict(X_train_padded)
X_test_features_cnn = feature_extractor.predict(X_test_padded)

print("Feature Extraction Successful!")
print("X_train_features shape:", X_train_features_cnn.shape)
print("X_test_features shape:", X_test_features_cnn.shape)

31/31 [==============================] - 0s 9ms/step
Feature Extraction Successful!
X_train_features shape: (2898, 128)
X_test_features shape: (967, 128)


### CNN: Bagging: Random Forest

In [ ]:
# Train Random Forest on CNN features
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_clf.fit(X_train_features_cnn, y_train)

# Predict and evaluate
rf_preds = rf_clf.predict(X_test_features_cnn)
print("Random Forest Results:")
print(f"Accuracy: {accuracy_score(y_test, rf_preds)}")
print(f"F1-Score: {f1_score(y_test, rf_preds)}")

Random Forest Results:
Accuracy: 0.7580144777662875
F1-Score: 0.727906976744186


### CNN: Boosting: XGBoost

In [ ]:
import xgboost as xgb
from xgboost import XGBClassifier

# Train XGBoost on CNN features
xgb_clf = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)
xgb_clf.fit(X_train_features_cnn, y_train)

# Predict and evaluate
xgb_preds = xgb_clf.predict(X_test_features_cnn)
print("XGBoost Results:")
print(f"Accuracy: {accuracy_score(y_test, xgb_preds)}")
print(f"F1-Score: {f1_score(y_test, xgb_preds)}")

/usr/local/lib/python3.10/dist-packages/xgboost/core.py:158: UserWarning: [08:55:26] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


XGBoost Results:
Accuracy: 0.7621509824198552
F1-Score: 0.7281323877068558


## Best Model: RNN with LSTM

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.metrics import accuracy_score, classification_report

best_params_lstm = {'lstm_units': 192, 'dropout_rate': 0.4, 'dense_units': 192, 'learning_rate': 0.0002717199130284055}
# Step 0: Preprocess Input Data
def preprocess_sentences(sentences):
    return [str(sentence) if isinstance(sentence, str) else "" for sentence in sentences]

X_train = preprocess_sentences(X_train)
X_val = preprocess_sentences(X_val)
X_test = preprocess_sentences(X_test)

# Step 1: Tokenize the text data
tokenizer = Tokenizer()
tokenizer.fit_on_texts(X_train)

# Convert texts to sequences of integers
X_train_sequences = tokenizer.texts_to_sequences(X_train)
X_val_sequences = tokenizer.texts_to_sequences(X_val)
X_test_sequences = tokenizer.texts_to_sequences(X_test)

# Step 2: Pad sequences
max_sequence_length = 100  # Set to a fixed size
X_train_padded = pad_sequences(X_train_sequences, maxlen=max_sequence_length)
X_val_padded = pad_sequences(X_val_sequences, maxlen=max_sequence_length)
X_test_padded = pad_sequences(X_test_sequences, maxlen=max_sequence_length)

# Step 3: Encode labels
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)
y_test_encoded = label_encoder.transform(y_test)

# Step 4: Build LSTM Model with Best Hyperparameters
model = Sequential([
    # Embedding Layer
    Embedding(input_dim=len(tokenizer.word_index) + 1,  # Vocabulary size
              output_dim=100,  # Embedding dimension
              input_length=max_sequence_length),

    # LSTM Layer
    LSTM(best_params_lstm['lstm_units'], return_sequences=False,
         dropout=best_params_lstm['dropout_rate'],
         recurrent_dropout=best_params_lstm['dropout_rate']),

    # Dense Layer
    Dense(best_params_lstm['dense_units'], activation='relu'),
    Dropout(best_params_lstm['dropout_rate']),

    # Output Layer
    Dense(len(np.unique(y_train_encoded)), activation='softmax')  # Adjust for the number of classes
])

# Compile the model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=best_params_lstm['learning_rate']),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Define callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)

# Step 5: Train the model
history = model.fit(
    X_train_padded, y_train_encoded,
    epochs=20,  # Adjust epochs as needed
    batch_size=64,
    validation_data=(X_val_padded, y_val_encoded),
    callbacks=[early_stopping, reduce_lr]
)

# Step 6: Evaluate on Test Set
y_test_pred = model.predict(X_test_padded)
y_test_pred_classes = np.argmax(y_test_pred, axis=1)

# Accuracy and Classification Report
accuracy = accuracy_score(y_test_encoded, y_test_pred_classes)
print(f"Test Accuracy: {accuracy:.4f}")
print("Classification Report:\n", classification_report(y_test_encoded, y_test_pred_classes))

Epoch 1/20
46/46 [==============================] - 63s 1s/step - loss: 0.6874 - accuracy: 0.5752 - val_loss: 0.6769 - val_accuracy: 0.6004 - lr: 2.7172e-04
Epoch 2/20
46/46 [==============================] - 44s 947ms/step - loss: 0.6334 - accuracy: 0.6632 - val_loss: 0.6433 - val_accuracy: 0.6304 - lr: 2.7172e-04
Epoch 3/20
46/46 [==============================] - 42s 926ms/step - loss: 0.5202 - accuracy: 0.7585 - val_loss: 0.5805 - val_accuracy: 0.7029 - lr: 2.7172e-04
Epoch 4/20
46/46 [==============================] - 39s 848ms/step - loss: 0.3401 - accuracy: 0.8627 - val_loss: 0.5544 - val_accuracy: 0.7329 - lr: 2.7172e-04
Epoch 5/20
46/46 [==============================] - 45s 994ms/step - loss: 0.2129 - accuracy: 0.9203 - val_loss: 0.5669 - val_accuracy: 0.7474 - lr: 2.7172e-04
Epoch 6/20
46/46 [==============================] - 42s 921ms/step - loss: 0.1350 - accuracy: 0.9548 - val_loss: 0.6153 - val_accuracy: 0.7547 - lr: 2.7172e-04
Epoch 7/20
46/46 [=========================

In [ ]:
model.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_4 (Embedding)     (None, 100, 100)          704100    
                                                                 
 lstm_1 (LSTM)               (None, 192)               225024    
                                                                 
 dense_4 (Dense)             (None, 192)               37056     
                                                                 
 dropout_1 (Dropout)         (None, 192)               0         
                                                                 
 dense_5 (Dense)             (None, 2)                 386       
                                                                 
Total params: 966,566
Trainable params: 966,566
Non-trainable params: 0
_________________________________________________________________


In [ ]:
from tensorflow.keras.models import Model
import numpy as np

# Step 1: Initialize the model
dummy_data = np.zeros((1, X_train_padded.shape[1]))  # Dummy input with the same shape as X_train_padded
model(dummy_data)  # Call the model to initialize its tensors

# Step 2: Extract features from the dense_4 layer
# Try accessing the dense_4 layer by name
try:
    feature_extractor_lstm = Model(inputs=model.input, outputs=model.get_layer("dense_4").output)
except ValueError:
    # If name access fails, access by index
    feature_extractor_lstm = Model(inputs=model.input, outputs=model.layers[-3].output)

# Step 3: Generate feature embeddings for train, validation, and test sets
X_train_features_lstm = feature_extractor_lstm.predict(X_train_padded)
X_val_features_lstm = feature_extractor_lstm.predict(X_val_padded)
X_test_features_lstm = feature_extractor_lstm.predict(X_test_padded)

# Print feature shapes
print(f"Train Features Shape: {X_train_features_lstm.shape}")
print(f"Validation Features Shape: {X_val_features_lstm.shape}")
print(f"Test Features Shape: {X_test_features_lstm.shape}")


31/31 [==============================] - 3s 82ms/step
Train Features Shape: (2898, 192)
Validation Features Shape: (966, 192)
Test Features Shape: (967, 192)


### LSTM: Bagging: Random Forest

In [ ]:
# Train Random Forest
rf_clf_lstm = RandomForestClassifier(n_estimators=100, random_state=42)
rf_clf_lstm.fit(X_train_features_lstm, y_train_encoded)

# Predict and evaluate
rf_preds_lstm = rf_clf_lstm.predict(X_test_features_lstm)
print("Random Forest Results:")
print(f"Accuracy: {accuracy_score(y_test_encoded, rf_preds_lstm)}")
print(f"F1-Score: {f1_score(y_test_encoded, rf_preds_lstm, average='weighted')}")

Random Forest Results:
Accuracy: 0.7776628748707343
F1-Score: 0.7783315737217329


### LSTM: Boosting: XGBoost

In [ ]:
# Train XGBoost
xgb_clf_lstm = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    use_label_encoder=False,
    eval_metric='mlogloss'
)
xgb_clf_lstm.fit(X_train_features_lstm, y_train_encoded)

# Predict and evaluate
xgb_preds_lstm = xgb_clf_lstm.predict(X_test_features_lstm)
print("XGBoost Results:")
print(f"Accuracy: {accuracy_score(y_test_encoded, xgb_preds_lstm)}")
print(f"F1-Score: {f1_score(y_test_encoded, xgb_preds_lstm, average='weighted')}")

/usr/local/lib/python3.10/dist-packages/xgboost/core.py:158: UserWarning: [09:05:28] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


XGBoost Results:
Accuracy: 0.7776628748707343
F1-Score: 0.7783423390362116


## Best Model: BERT

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import Trainer, TrainingArguments
from transformers import EarlyStoppingCallback
import os
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

best_params_bert = {'learning_rate': 7.545969011015214e-05, 'num_train_epochs': 3, 'per_device_train_batch_size': 32, 'per_device_eval_batch_size': 16, 'weight_decay': 0.0009563874807305245}

# Disable W&B logging
os.environ["WANDB_DISABLED"] = "true"

# Load tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased',
                                          hidden_dropout_prob=0.3,
                                          attention_probs_dropout_prob=0.3)  # Combat overfitting
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# Tokenize data
train_encodings = tokenizer(list(X), truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(list(X_test), truncation=True, padding=True, max_length=128)

# Create Dataset class
class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

# Create train and test datasets
train_dataset = Dataset(train_encodings, list(y))
test_dataset = Dataset(test_encodings, list(y_test))

# Define training arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=best_params_bert['num_train_epochs'],
    per_device_train_batch_size=best_params_bert['per_device_train_batch_size'],
    per_device_eval_batch_size=best_params_bert['per_device_eval_batch_size'],
    warmup_steps=100,
    weight_decay=best_params_bert['weight_decay'],
    logging_dir='./logs',
    logging_steps=10,
    evaluation_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    learning_rate=best_params_bert['learning_rate']  # Value found from optuna tuning
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],  # Early stopping
)

# Train the model
trainer.train()

# Evaluate the model
eval_results = trainer.evaluate(test_dataset)
print(f"Evaluation Results: {eval_results}")

# Predict and evaluate metrics
predictions = trainer.predict(test_dataset)
preds = predictions.predictions.argmax(axis=1)  # Predicted labels
labels = predictions.label_ids  # True labels

# Calculate accuracy
accuracy = accuracy_score(labels, preds)
print(f"Accuracy: {accuracy}")

# Print classification report
class_report = classification_report(labels, preds, target_names=["Class 0", "Class 1"])  # Update target names as needed
print("Classification Report:")
print(class_report)

# Print confusion matrix
conf_matrix = confusion_matrix(labels, preds)
print("Confusion Matrix:")
print(conf_matrix)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Step,Training Loss,Validation Loss


Evaluation Results: {'eval_loss': 0.49586424231529236, 'eval_runtime': 4.608, 'eval_samples_per_second': 209.85, 'eval_steps_per_second': 13.238, 'epoch': 3.0}
Accuracy: 0.796277145811789
Classification Report:
              precision    recall  f1-score   support

     Class 0       0.85      0.77      0.81       537
     Class 1       0.74      0.83      0.78       430

    accuracy                           0.80       967
   macro avg       0.80      0.80      0.80       967
weighted avg       0.80      0.80      0.80       967

Confusion Matrix:
[[413 124]
 [ 73 357]]


In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import torch

# Function to extract embeddings from the fine-tuned BERT model
def get_fine_tuned_bert_embeddings(texts, tokenizer, model, max_length=128):
    model.eval()
    embeddings = []
    with torch.no_grad():
        for text in texts:
            inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=max_length)
            inputs = {key: val.to(model.device) for key, val in inputs.items()}  # Move inputs to the same device as the model
            outputs = model(**inputs)
            cls_embedding = outputs.hidden_states[-1][:, 0, :].squeeze().cpu().numpy()  # CLS token embeddings
            embeddings.append(cls_embedding)
    return np.array(embeddings)

# Ensure your model outputs hidden states
model.config.output_hidden_states = True

# Extract embeddings from the fine-tuned BERT model
print("Extracting fine-tuned BERT embeddings for training data...")
train_embeddings = get_fine_tuned_bert_embeddings(X, tokenizer, model)

print("Extracting fine-tuned BERT embeddings for test data...")
test_embeddings = get_fine_tuned_bert_embeddings(X_test, tokenizer, model)

Extracting fine-tuned BERT embeddings for training data...
Extracting fine-tuned BERT embeddings for test data...


### BERT: Bagging: Random Forest

In [ ]:
# Train Random Forest
print("Training Random Forest...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(train_embeddings, y)

# Evaluate Random Forest
rf_preds = rf_model.predict(test_embeddings)
rf_accuracy = accuracy_score(y_test, rf_preds)
print(f"Random Forest Accuracy: {rf_accuracy}")
print("Random Forest Classification Report:")
print(classification_report(y_test, rf_preds, target_names=["Class 0", "Class 1"]))
print("Random Forest Confusion Matrix:")
print(confusion_matrix(y_test, rf_preds))

Training Random Forest...
Random Forest Accuracy: 0.8045501551189245
Random Forest Classification Report:
              precision    recall  f1-score   support

     Class 0       0.83      0.82      0.82       537
     Class 1       0.78      0.79      0.78       430

    accuracy                           0.80       967
   macro avg       0.80      0.80      0.80       967
weighted avg       0.80      0.80      0.80       967

Random Forest Confusion Matrix:
[[439  98]
 [ 91 339]]


### BERT: Boosting: XGBoost

In [ ]:
# Train XGBoost
print("Training XGBoost...")
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_model.fit(train_embeddings, y)

# Evaluate XGBoost
xgb_preds = xgb_model.predict(test_embeddings)
xgb_accuracy = accuracy_score(y_test, xgb_preds)
print(f"XGBoost Accuracy: {xgb_accuracy}")
print("XGBoost Classification Report:")
print(classification_report(y_test, xgb_preds, target_names=["Class 0", "Class 1"]))
print("XGBoost Confusion Matrix:")
print(confusion_matrix(y_test, xgb_preds))

Training XGBoost...


/usr/local/lib/python3.10/dist-packages/xgboost/core.py:158: UserWarning: [01:59:48] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


XGBoost Accuracy: 0.7921406411582212
XGBoost Classification Report:
              precision    recall  f1-score   support

     Class 0       0.82      0.80      0.81       537
     Class 1       0.76      0.78      0.77       430

    accuracy                           0.79       967
   macro avg       0.79      0.79      0.79       967
weighted avg       0.79      0.79      0.79       967

XGBoost Confusion Matrix:
[[432 105]
 [ 96 334]]


# Upload model to huggingface

In [ ]:
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: fineGrained).
The token `ML5190Final` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `ML5190

In [ ]:
from transformers import BertConfig, BertModel

config = BertConfig()
model = BertModel(config) # parameters to tune

model.push_to_hub("elawrie/my-awesome-bert-model")

# reload
model = BertModel.from_pretrained("elawrie/my-awesome-bert-model")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

#Sentiment analysis
Identify the average sentiment of NBC and Fox articles, respectively, using VADER.

First, split the data into NBC and Fox datasets for separate sentiment analysis

In [ ]:
# separate train and test set by source
data = {"title": [], "source": []}
nbc_df = pd.DataFrame(data)
fox_df = pd.DataFrame(data)

sources = train_df["source"]
index = 0

for line in sources:
  line_df = pd.DataFrame({"title": train_df["title_lemmatized"][index], "source": [line]})
  if line == 1:
    nbc_df = pd.concat([nbc_df, line_df], ignore_index=True)
  else:
    fox_df = pd.concat([fox_df, line_df], ignore_index=True)
  index += 1

In [ ]:
sources2 = test_df["source"]
index2 = 0

for line in sources2:
  line_df = pd.DataFrame({"title": train_df["title_lemmatized"][index2], "source": [line]})
  if line == 1:
    nbc_df = pd.concat([nbc_df, line_df], ignore_index=True)
  else:
    fox_df = pd.concat([fox_df, line_df], ignore_index=True)
  index2 += 1

In [ ]:
nbc_df.head()

,title,source
0,7 face wash acneprone skin year,1.0
1,thanksgiving dinner historically affordable year,1.0
2,floridas surgeon general advise add fluoride d...,1.0
3,former nfl qb jay cutler arrest charge dui gun...,1.0
4,bestselling tech purchase tech products cover ...,1.0


In [ ]:
fox_df.head()

,title,source
0,meet american inspire nation two world war chr...,0.0
1,politico alter headline call kamala harris sup...,0.0
2,honey deuce sales us open reveal cocktail expl...,0.0
3,apalachee high school shoot georgia assistant ...,0.0
4,vietnam war formally end day history,0.0


In [ ]:
# print size of each dataframe
nbc_df.shape[0]

2233

In [ ]:
fox_df.shape[0]

2598

In [ ]:
pip install vaderSentiment

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 5.4 MB/s eta 0:00:00


In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
sentiment = SentimentIntensityAnalyzer()

# loop through all nbc articles and calculate average polarity
total_polarity = 0
num_articles = nbc_df.shape[0]

polarity_vals = []
polarity_conts = []

for article in nbc_df["title"]:
  sent_1 = sentiment.polarity_scores(article)
  total_polarity += sent_1["compound"]
  # make the polarity value categorical
  # a sentiment is considered positive if the "compound" score is greater than or equal to 0.05,
  # negative if it's less than or equal to -0.05, and
  # neutral if it falls between those two values (between -0.05 and 0.05)
  polarity_conts.append(sent_1["compound"])
  if sent_1["compound"] >= 0.05:
    polarity_vals.append(1)
  elif sent_1["compound"] <= -0.05:
    polarity_vals.append(-1)
  else:
    polarity_vals.append(0)

avg_polarity = total_polarity / num_articles
print("Average polarity of NBC articles:", avg_polarity)

# add a new column to the dataframes with the sentiment of the title
nbc_df["polarity"] = polarity_vals
nbc_df["polarity_cont"] = polarity_conts

Average polarity of NBC articles: -0.056506493506493546


In [ ]:
nbc_df.head()

,title,source,polarity,polarity_cont
0,7 face wash acneprone skin year,1.0,0,0.0000
1,thanksgiving dinner historically affordable year,1.0,0,0.0000
2,floridas surgeon general advise add fluoride d...,1.0,0,0.0000
3,former nfl qb jay cutler arrest charge dui gun...,1.0,-1,-0.7579
4,bestselling tech purchase tech products cover ...,1.0,0,0.0000


In [ ]:
# get average polarity of fox articles
total_polarity = 0
num_articles = fox_df.shape[0]

polarity_vals2 = []
polarity_conts2 = []

for article in fox_df["title"]:
  sent_1 = sentiment.polarity_scores(article)
  total_polarity += sent_1["compound"]
  # make the polarity value categorical
  # a sentiment is considered positive if the "compound" score is greater than or equal to 0.05,
  # negative if it's less than or equal to -0.05, and
  # neutral if it falls between those two values (between -0.05 and 0.05)
  polarity_conts2.append(sent_1["compound"])
  if sent_1["compound"] >= 0.05:
    polarity_vals2.append(1)
  elif sent_1["compound"] <= -0.05:
    polarity_vals2.append(-1)
  else:
    polarity_vals2.append(0)

avg_polarity = total_polarity / num_articles
print("Average polarity of Fox articles:", avg_polarity)

# add a new column to the dataframes with the sentiment of the title
fox_df["polarity"] = polarity_vals2
fox_df["polarity_cont"] = polarity_conts2


Average polarity of Fox articles: -0.07736978444957647


In [ ]:
fox_df.head()

,title,source,polarity,polarity_cont
0,meet american inspire nation two world war chr...,0.0,-1,-0.0516
1,politico alter headline call kamala harris sup...,0.0,-1,-0.4902
2,honey deuce sales us open reveal cocktail expl...,0.0,1,0.4767
3,apalachee high school shoot georgia assistant ...,0.0,-1,-0.7717
4,vietnam war formally end day history,0.0,-1,-0.5994


In [ ]:
# calculate polarity of training data
polarity_vals3 = []

for article in train_df["title_lemmatized"]:
  sent_1 = sentiment.polarity_scores(article)
  polarity_vals3.append(sent_1["compound"])

# add a new column to the dataframe with the sentiment of the title
train_df["polarity"] = polarity_vals3

In [ ]:
train_df.head()

,title_lemmatized,source,polarity
0,7 face wash acneprone skin year,1,0.0000
1,thanksgiving dinner historically affordable year,1,0.0000
2,floridas surgeon general advise add fluoride d...,1,0.0000
3,meet american inspire nation two world war chr...,0,-0.0516
4,former nfl qb jay cutler arrest charge dui gun...,1,-0.7579


In [ ]:
# calculate polarity of testing data
polarity_vals4 = []

for article in test_df["title_lemmatized"]:
  sent_1 = sentiment.polarity_scores(article)
  polarity_vals4.append(sent_1["compound"])

# add a new column to the dataframe with the sentiment of the title
test_df["polarity"] = polarity_vals4

In [ ]:
test_df.head()

,title_lemmatized,source,polarity
0,kamala harris reassure democratic party donors...,0,0.7584
1,serious crisis grip america one want talk,0,-0.6249
2,nurse students use virtual reality enhance ski...,0,0.5106
3,modi lose magic — majority — india election su...,1,-0.1531
4,become nurse covid former producer double hour...,1,0.0000


Fox titles are slightly more negative on average than NBC titles. However, both sources have on average pretty negative sentiments (with NBC's average just being on the edge of consideration for a negative sentiment, which is -0.05). This is given that VADER produces a sentiment score from -1 (negative) to 1 (positive).

# Sentiment Predictions
It is also possible to train a classifer to predict the sentiment of a Fox or NBC title.

In [ ]:
# NBC sentiment prediction using bag of words

# pre-prcoess and Bag of Word Vectorization using Count Vectorizer
from sklearn.feature_extraction.text import CountVectorizer
from nltk.tokenize import RegexpTokenizer
token = RegexpTokenizer(r'[a-zA-Z0-9]+')
cv = CountVectorizer(stop_words='english',ngram_range = (1,1),tokenizer = token.tokenize)
text_counts = cv.fit_transform(nbc_df['title'])

# split the data into trainig and testing
from sklearn.model_selection import train_test_split
X_train1, X_test1, Y_train1, Y_test1 = train_test_split(text_counts, nbc_df['polarity'], test_size=0.25, random_state=5)

# train the model
from sklearn.naive_bayes import MultinomialNB
MNB = MultinomialNB()
MNB.fit(X_train1, Y_train1)

# caluclate the accuracy score of the model
from sklearn import metrics
predicted = MNB.predict(X_test1)
accuracy_score = metrics.accuracy_score(predicted, Y_test1)
print("Accuracy Score: ",accuracy_score)

/usr/local/lib/python3.10/dist-packages/sklearn/feature_extraction/text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Accuracy Score:  0.6905187835420393


In [ ]:
# Fox sentiment prediction using bag of words

# pre-prcoess and Bag of Word Vectorization using Count Vectorizer
from sklearn.feature_extraction.text import CountVectorizer
from nltk.tokenize import RegexpTokenizer
token = RegexpTokenizer(r'[a-zA-Z0-9]+')
cv = CountVectorizer(stop_words='english',ngram_range = (1,1),tokenizer = token.tokenize)
text_counts = cv.fit_transform(fox_df['title'])

# split the data into trainig and testing
from sklearn.model_selection import train_test_split
X_train2, X_test2, Y_train2, Y_test2 = train_test_split(text_counts, fox_df['polarity'], test_size=0.25, random_state=5)

# train the model
from sklearn.naive_bayes import MultinomialNB
MNB = MultinomialNB()
MNB.fit(X_train2, Y_train2)

# caluclate the accuracy score of the model
from sklearn import metrics
predicted = MNB.predict(X_test2)
accuracy_score = metrics.accuracy_score(predicted, Y_test2)
print("Accuracy Score: ",accuracy_score)

/usr/local/lib/python3.10/dist-packages/sklearn/feature_extraction/text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Accuracy Score:  0.6692307692307692


Bag of words models predict the sentiment scores of NBC titles with higher accuracy (69%) than Fox articles (66%).

Now try predicting sentiments with an LSTM model.

In [ ]:
pip install keras

In [ ]:
pip install keras-preprocessing

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 3.7 MB/s eta 0:00:00


In [ ]:
# predict NBC sentiments

import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
import nltk
import pandas as pd
from textblob import Word
from nltk.corpus import stopwords
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report,confusion_matrix,accuracy_score
from keras.models import Sequential
from keras.layers import TextVectorization
from keras_preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from keras.layers import Dense, Embedding, LSTM, SpatialDropout1D, Dropout

# preprocessing
max_sequence_length = 18

X_train3, X_test3, y_train3, y_test3 = train_test_split(
    nbc_df['title'], nbc_df['polarity_cont'], test_size=0.25, random_state=5
)

nbc_tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
nbc_tokenizer.fit_on_texts(X_train3)

# tokenize and pad sequences
training_sequences = nbc_tokenizer.texts_to_sequences(X_train3)
training_padded = pad_sequences(
    training_sequences, maxlen=max_sequence_length, padding='post', truncating='post'
)
testing_sequences = nbc_tokenizer.texts_to_sequences(X_test3)
testing_padded = pad_sequences(
    testing_sequences, maxlen=max_sequence_length, padding='post', truncating='post'
)

# convert to numpy arrays
training_padded = np.array(training_padded)
training_labels = np.array(y_train3, dtype=np.float32)
testing_padded = np.array(testing_padded)
testing_labels = np.array(y_test3, dtype=np.float32)

# ensure labels are binary (if needed)
training_labels = np.where(training_labels > 0, 1, 0).astype(np.float32)
testing_labels = np.where(testing_labels > 0, 1, 0).astype(np.float32)

# define the model
nbc_model = tf.keras.Sequential([
    tf.keras.layers.Embedding(10000, 16, input_length=max_sequence_length),  # input_length = 18
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(24, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')  # Binary classification
])

# explicitly build the model
nbc_model.build(input_shape=(None, max_sequence_length))  # Force initialization
print(nbc_model.summary())

# Compile the model
nbc_model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

num_epochs = 10
history = nbc_model.fit(training_padded,
                    training_labels,
                    epochs=num_epochs,
                    validation_data=(testing_padded, testing_labels),
                    verbose=2)

# evaluate the model
loss, accuracy = nbc_model.evaluate(testing_padded, testing_labels)
print(f"Test accuracy: {accuracy:.4f}")
# print classification report
y_pred = (nbc_model.predict(testing_padded) > 0.5).astype(int)
print(classification_report(testing_labels, y_pred))

/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ (None, 18, 16)              │         160,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling1d             │ (None, 16)                  │               0 │
│ (GlobalAveragePooling1D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 24)                  │             408 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │              25 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 160,433 (626.69 KB)

 Trainable params: 160,433 (626.69 KB)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/10
53/53 - 9s - 179ms/step - accuracy: 0.6971 - loss: 0.6458 - val_accuracy: 0.6744 - val_loss: 0.6242
Epoch 2/10
53/53 - 0s - 7ms/step - accuracy: 0.6983 - loss: 0.6046 - val_accuracy: 0.6744 - val_loss: 0.6173
Epoch 3/10
53/53 - 0s - 6ms/step - accuracy: 0.6983 - loss: 0.5903 - val_accuracy: 0.6744 - val_loss: 0.6066
Epoch 4/10
53/53 - 0s - 4ms/step - accuracy: 0.6983 - loss: 0.5677 - val_accuracy: 0.6744 - val_loss: 0.5855
Epoch 5/10
53/53 - 0s - 4ms/step - accuracy: 0.7019 - loss: 0.5243 - val_accuracy: 0.6887 - val_loss: 0.5494
Epoch 6/10
53/53 - 0s - 4ms/step - accuracy: 0.7658 - loss: 0.4541 - val_accuracy: 0.7621 - val_loss: 0.5015
Epoch 7/10
53/53 - 0s - 6ms/step - accuracy: 0.8405 - loss: 0.3659 - val_accuracy: 0.7782 - val_loss: 0.4555
Epoch 8/10
53/53 - 0s - 5ms/step - accuracy: 0.9086 - loss: 0.2740 - val_accuracy: 0.8175 - val_loss: 0.4101
Epoch 9/10
53/53 - 0s - 7ms/step - accuracy: 0.9492 - loss: 0.1977 - val_accuracy: 0.8354 - val_loss: 0.3862
Epoch 10/10


In [ ]:
# predict Fox sentiments

# preprocessing
max_sequence_length = 18

X_train4, X_test4, y_train4, y_test4 = train_test_split(fox_df['title'], fox_df['polarity_cont'], test_size=0.25, random_state=5)

fox_tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
fox_tokenizer.fit_on_texts(X_train4)

# tokenize and pad sequences
training_sequences2 = fox_tokenizer.texts_to_sequences(X_train4)
training_padded2 = pad_sequences(
    training_sequences2, maxlen=max_sequence_length, padding='post', truncating='post'
)
testing_sequences2 = fox_tokenizer.texts_to_sequences(X_test4)
testing_padded2 = pad_sequences(
    testing_sequences2, maxlen=max_sequence_length, padding='post', truncating='post'
)

# convert to numpy arrays
training_padded2 = np.array(training_padded2)
training_labels2 = np.array(y_train4, dtype=np.float32)
testing_padded2 = np.array(testing_padded2)
testing_labels2 = np.array(y_test4, dtype=np.float32)

# ensure labels are binary (if needed)
training_labels2 = np.where(training_labels2 > 0, 1, 0).astype(np.float32)
testing_labels2 = np.where(testing_labels2 > 0, 1, 0).astype(np.float32)

# define the model
fox_model = tf.keras.Sequential([
    tf.keras.layers.Embedding(10000, 16, input_length=max_sequence_length),  # input_length = 18
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(24, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')  # Binary classification
])

# explicitly build the model
fox_model.build(input_shape=(None, max_sequence_length))  # Force initialization
print(fox_model.summary())

# compile the model
fox_model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

num_epochs = 10
history = fox_model.fit(training_padded2,
                    training_labels2,
                    epochs=num_epochs,
                    validation_data=(testing_padded2, testing_labels2),
                    verbose=2)

# evaluate the model
loss, accuracy = fox_model.evaluate(testing_padded2, testing_labels2)
print(f"Test accuracy: {accuracy:.4f}")
# print classification report
y_pred2 = (fox_model.predict(testing_padded2) > 0.5).astype(int)
print(classification_report(testing_labels2, y_pred2))


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)              │ (None, 18, 16)              │         160,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling1d_1           │ (None, 16)                  │               0 │
│ (GlobalAveragePooling1D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 24)                  │             408 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 1)                   │              25 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 160,433 (626.69 KB)

 Trainable params: 160,433 (626.69 KB)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/10
61/61 - 5s - 89ms/step - accuracy: 0.6797 - loss: 0.6440 - val_accuracy: 0.6877 - val_loss: 0.6157
Epoch 2/10
61/61 - 0s - 7ms/step - accuracy: 0.6822 - loss: 0.6139 - val_accuracy: 0.6877 - val_loss: 0.6097
Epoch 3/10
61/61 - 0s - 2ms/step - accuracy: 0.6822 - loss: 0.6010 - val_accuracy: 0.6877 - val_loss: 0.6013
Epoch 4/10
61/61 - 0s - 3ms/step - accuracy: 0.6828 - loss: 0.5793 - val_accuracy: 0.6877 - val_loss: 0.5850
Epoch 5/10
61/61 - 0s - 2ms/step - accuracy: 0.6961 - loss: 0.5291 - val_accuracy: 0.7431 - val_loss: 0.5517
Epoch 6/10
61/61 - 0s - 5ms/step - accuracy: 0.8229 - loss: 0.4251 - val_accuracy: 0.8031 - val_loss: 0.4760
Epoch 7/10
61/61 - 0s - 3ms/step - accuracy: 0.9286 - loss: 0.2951 - val_accuracy: 0.8200 - val_loss: 0.4172
Epoch 8/10
61/61 - 0s - 2ms/step - accuracy: 0.9687 - loss: 0.1927 - val_accuracy: 0.8292 - val_loss: 0.3858
Epoch 9/10
61/61 - 0s - 5ms/step - accuracy: 0.9825 - loss: 0.1280 - val_accuracy: 0.8308 - val_loss: 0.3732
Epoch 10/10
6

Using sentiment scores to predict source of headline text

First, compute sentiment scores for all headlines using the models above

In [ ]:
# Tokenize and pad the sequences for NBC and Fox separately
nbc_sequences = nbc_tokenizer.texts_to_sequences(train_df['title_lemmatized'])
nbc_padded = pad_sequences(nbc_sequences, maxlen=max_sequence_length, padding='post', truncating='post')

fox_sequences = fox_tokenizer.texts_to_sequences(train_df['title_lemmatized'])
fox_padded = pad_sequences(fox_sequences, maxlen=max_sequence_length, padding='post', truncating='post')

# Define a function to compute sentiment
def compute_sentiment(row):
    if row['source'] == 1:  # NBC
        return nbc_model.predict(np.expand_dims(nbc_padded[row.name], axis=0)).squeeze()
    elif row['source'] == 0:  # Fox
        return fox_model.predict(np.expand_dims(fox_padded[row.name], axis=0)).squeeze()

Use the

In [ ]:
train_df['sentiment_score'] = train_df.apply(compute_sentiment, axis=1)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 481ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 196ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━

In [ ]:
train_df.head()

,title_lemmatized,source,polarity,sentiment_score
0,7 face wash acneprone skin year,1,0.0000,0.32830942
1,thanksgiving dinner historically affordable year,1,0.0000,0.20165032
2,floridas surgeon general advise add fluoride d...,1,0.0000,0.05980768
3,meet american inspire nation two world war chr...,0,-0.0516,0.2242337
4,former nfl qb jay cutler arrest charge dui gun...,1,-0.7579,0.0061362544


In [ ]:
test_df['sentiment_score'] = test_df.apply(compute_sentiment, axis=1)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━

In [ ]:
test_df.head()

,title_lemmatized,source,polarity,sentiment_score
0,kamala harris reassure democratic party donors...,0,0.7584,0.29461324
1,serious crisis grip america one want talk,0,-0.6249,0.121420495
2,nurse students use virtual reality enhance ski...,0,0.5106,0.025004985
3,modi lose magic — majority — india election su...,1,-0.1531,0.015673622
4,become nurse covid former producer double hour...,1,0.0000,0.0061362544


Then split into training and testing data using the sentiment scores as the features for classification

Test predictions using the VADER sentiment scores (polarity)

In [ ]:
# Training features: only the sentiment score
X_train5 = train_df[['polarity']]
y_train5 = train_df['source']

# Testing features: only the sentiment score
X_test5 = test_df[['polarity']]
y_test5 = test_df['source']

Predict with a basic logistic regression model first

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Initialize and train logistic regression
log_reg = LogisticRegression()
log_reg.fit(X_train5, y_train5)

# Predict on test data
y_pred = log_reg.predict(X_test5)

# Evaluate
print(classification_report(y_test5, y_pred))

              precision    recall  f1-score   support

           0       0.56      1.00      0.71       537
           1       1.00      0.00      0.00       430

    accuracy                           0.56       967
   macro avg       0.78      0.50      0.36       967
weighted avg       0.75      0.56      0.40       967



Predict with a random forest classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Train Random Forest Classifier
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train5, y_train5)

# Predict and evaluate
y_pred = rf_model.predict(X_test5)
print(classification_report(y_test5, y_pred))

              precision    recall  f1-score   support

           0       0.58      0.56      0.57       537
           1       0.48      0.50      0.49       430

    accuracy                           0.53       967
   macro avg       0.53      0.53      0.53       967
weighted avg       0.54      0.53      0.54       967



Test predictions with the NBC and Fox models (sentiment_score)

In [ ]:
# training features: only the sentiment score
X_train6 = train_df[['sentiment_score']]
y_train6 = train_df['source']

# testing features: only the sentiment score
X_test6 = test_df[['sentiment_score']]
y_test6 = test_df['source']

Use a basic logistic regression model first

In [ ]:
# nitialize and train logistic regression
log_reg2 = LogisticRegression()
log_reg2.fit(X_train6, y_train6)

# predict on test data
y_pred2 = log_reg2.predict(X_test6)

# evaluate
print(classification_report(y_test6, y_pred2))

              precision    recall  f1-score   support

           0       0.56      1.00      0.71       537
           1       0.00      0.00      0.00       430

    accuracy                           0.56       967
   macro avg       0.28      0.50      0.36       967
weighted avg       0.31      0.56      0.40       967



/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Next, predict with random forest

In [ ]:
# train Random Forest Classifier
rf_model2 = RandomForestClassifier(random_state=42)
rf_model2.fit(X_train6, y_train6)

# predict and evaluate
y_pred2 = rf_model2.predict(X_test6)
print(classification_report(y_test6, y_pred2))

              precision    recall  f1-score   support

           0       0.80      0.82      0.81       537
           1       0.77      0.74      0.75       430

    accuracy                           0.78       967
   macro avg       0.78      0.78      0.78       967
weighted avg       0.78      0.78      0.78       967

